## Using ATP Match Data from 2018-2026 To Predict Wimbledon Bracket Results

## NOTE: ON 25th JUNEish UPDATE 2026 FILE WITH UPDATED RECENT TOURNAMENT RESULTS

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Creating dataframe of matches in chronological order (01/01/2018-06/14/2026)

In [19]:
file_paths_ordered = ["2018.csv", "2019.csv", "2020.csv", "2021.csv", "2022.csv", "2023.csv", "2024.csv", "2025.csv", "2026.csv"]

df_list = [pd.read_csv(file) for file in file_paths_ordered]
match_df = pd.concat(df_list, ignore_index=True)


match_df["tourney_date"] = pd.to_datetime(match_df["tourney_date"].astype(str),
    format="%Y%m%d"
)
match_df = match_df.sort_values(
    ["tourney_date", "tourney_id", "match_num"]
).reset_index(drop=True)

match_df[["tourney_date", "tourney_name", "surface", "winner_name", "loser_name", "score"]].head()

match_df.head()


,tourney_id,tourney_name,surface,draw_size,tourney_level,indoor,tourney_date,match_num,winner_id,winner_seed,...,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced
0,2018-339,Brisbane,Hard,32.0,250,O,2018-01-01,1.0,MH30,NaN,...,1.0,1.0,1.0,74.0,43.0,24.0,17.0,9.0,7.0,10.0
1,2018-339,Brisbane,Hard,32.0,250,O,2018-01-01,2.0,E831,NaN,...,6.0,18.0,5.0,100.0,62.0,54.0,18.0,17.0,1.0,3.0
2,2018-339,Brisbane,Hard,32.0,250,O,2018-01-01,3.0,CH27,NaN,...,2.0,19.0,5.0,68.0,40.0,33.0,10.0,11.0,3.0,5.0
3,2018-339,Brisbane,Hard,32.0,250,O,2018-01-01,4.0,E690,NaN,...,3.0,4.0,4.0,46.0,25.0,18.0,6.0,8.0,1.0,5.0
4,2018-339,Brisbane,Hard,32.0,250,O,2018-01-01,5.0,Z184,NaN,...,1.0,9.0,0.0,51.0,32.0,25.0,10.0,9.0,4.0,6.0


# Regular ELO Engineering:

ELO - Rating system: "Based on past results, how strong is this player coming into this match?"

Using ELO as opposed to just ATP rankings for surface specific ratings

Every player begins with rating = 1500

Suppose Player A has rating $R_A$ and Player B has rating $R_B$. 

Elo says:

$P$(A beats B) = $\frac{1}{{1+10^{{(R_B - R_A)}/400}}}$

As rating difference increases, so does win probability. Rating difference of 0 is 50$ win probability. Rating difference of 400 is 91%.

Updates:

new rating = old rating + $K$ $\times$ (actual result - expected result)

Where $K$ controls how fast ratings move.

Overall ELO uses all of a player's matches, whereas surface ELO uses only matches played on that surface. This is especially important for grass.

In [28]:
def expected_win_prob(rating_a, rating_b):
    return 1/(1+10**((rating_b-rating_a)/400))

def add_simple_elo_features(df, k=32, start_elo=1500):
    """
    Adds pre-match Elo ratings for winner and loser.
    
    Parameters:
        df: match dataframe sorted chronologically
        k: Elo update speed
        start_elo: rating assigned to players when first seen
    
    Returns:
        df with w_elo_pre and l_elo_pre columns
    """
    
    df = df.copy()
    
    # Stores current Elo rating for each player
    ratings = {}
    
    # Lists to store pre-match ratings
    w_elo_pre = []
    l_elo_pre = []
    
    for _, row in df.iterrows():
        winner = row["winner_id"]
        loser = row["loser_id"]
        
        # Get current ratings, or assign start_elo if player has never appeared
        winner_rating = ratings.get(winner, start_elo)
        loser_rating = ratings.get(loser, start_elo)
        
        # Store pre-match ratings
        w_elo_pre.append(winner_rating)
        l_elo_pre.append(loser_rating)
        
        # Expected probability that the winner would beat the loser
        expected_winner = expected_win_prob(winner_rating, loser_rating)
        expected_loser = 1 - expected_winner
        
        # Update ratings after the match
        ratings[winner] = winner_rating + k * (1 - expected_winner)
        ratings[loser] = loser_rating + k * (0 - expected_loser)
    
    df["w_elo_pre"] = w_elo_pre
    df["l_elo_pre"] = l_elo_pre
    
    return df

In [29]:
match_df = add_simple_elo_features(match_df, k=32, start_elo = 1500)

match_df[["tourney_date", "tourney_name", "winner_name", "loser_name", "w_elo_pre", "l_elo_pre"]]

,tourney_date,tourney_name,winner_name,loser_name,w_elo_pre,l_elo_pre
0,2018-01-01,Brisbane,John Millman,Peter Polansky,1500.000000,1500.000000
1,2018-01-01,Brisbane,Kyle Edmund,Denis Shapovalov,1500.000000,1500.000000
2,2018-01-01,Brisbane,Hyeon Chung,Gilles Muller,1500.000000,1500.000000
3,2018-01-01,Brisbane,Matthew Ebden,Frances Tiafoe,1500.000000,1500.000000
4,2018-01-01,Brisbane,Horacio Zeballos,Ernesto Escobedo,1500.000000,1500.000000
...,...,...,...,...,...,...
23319,2026-06-13,'s-Hertogenbosch,Alex De Minaur,Adrian Mannarino,1825.161224,1558.744009
23320,2026-06-13,'s-Hertogenbosch,Daniil Medvedev,Marin Cilic,1867.144392,1642.529856
23321,2026-06-13,'s-Hertogenbosch,Kamil Majchrzak,Daniil Medvedev,1636.017388,1874.035527
23322,2026-06-14,Stuttgart,Ben Shelton,Taylor Fritz,1796.809910,1781.029744


Create df that randomizes order of winner and loser columns

In [38]:
np.random.seed(42)

winner_is_A = np.random.rand(len(match_df)) < 0.5

neutral_df = pd.DataFrame({
    "date": match_df["tourney_date"],
    "tourney_id": match_df["tourney_id"],
    "tourney_name": match_df["tourney_name"],
    "surface": match_df["surface"],
    "draw_size": match_df["draw_size"],
    "tourney_level": match_df["tourney_level"],
    "indoor": match_df["indoor"],
    "round": match_df["round"],
    "best_of": match_df["best_of"],
    
    # Player A info
    "player_A_id": np.where(winner_is_A, match_df["winner_id"], match_df["loser_id"]),
    "player_A_name": np.where(winner_is_A, match_df["winner_name"], match_df["loser_name"]),
    "player_A_rank": np.where(winner_is_A, match_df["winner_rank"], match_df["loser_rank"]),
    "player_A_rank_points": np.where(winner_is_A, match_df["winner_rank_points"], match_df["loser_rank_points"]),
    "player_A_age": np.where(winner_is_A, match_df["winner_age"], match_df["loser_age"]),
    "player_A_ht": np.where(winner_is_A, match_df["winner_ht"], match_df["loser_ht"]),
    "player_A_hand": np.where(winner_is_A, match_df["winner_hand"], match_df["loser_hand"]),
    "player_A_elo_pre": np.where(winner_is_A, match_df["w_elo_pre"], match_df["l_elo_pre"]),

    # Player B info
    "player_B_id": np.where(winner_is_A, match_df["loser_id"], match_df["winner_id"]),
    "player_B_name": np.where(winner_is_A, match_df["loser_name"], match_df["winner_name"]),
    "player_B_rank": np.where(winner_is_A, match_df["loser_rank"], match_df["winner_rank"]),
    "player_B_rank_points": np.where(winner_is_A, match_df["loser_rank_points"], match_df["winner_rank_points"]),
    "player_B_age": np.where(winner_is_A, match_df["loser_age"], match_df["winner_age"]),
    "player_B_ht": np.where(winner_is_A, match_df["loser_ht"], match_df["winner_ht"]),
    "player_B_hand": np.where(winner_is_A, match_df["loser_hand"], match_df["winner_hand"]),
    "player_B_elo_pre": np.where(winner_is_A, match_df["l_elo_pre"], match_df["w_elo_pre"]),

    # Target
    "A_won": winner_is_A.astype(int)
})

neutral_df["elo_diff"] = (
    neutral_df["player_A_elo_pre"] - neutral_df["player_B_elo_pre"]
)

neutral_df[[
    "date",
    "tourney_name",
    "player_A_name",
    "player_B_name",
    "player_A_elo_pre",
    "player_B_elo_pre",
    "elo_diff",
    "A_won"
]].tail(10)

neutral_df["A_higher_elo"] = neutral_df["elo_diff"] > 0

neutral_df.groupby("A_higher_elo")["A_won"].mean()

A_higher_elo
False    0.368461
True     0.628858
Name: A_won, dtype: float64

Split data into train and test, with train being matches from 2018-2025 and test being matches from 2025 onwards

In [41]:
elo_model_df = neutral_df.dropna(subset=["elo_diff", "A_won"]).copy()

train = elo_model_df[elo_model_df["date"] < "2025-01-01"].copy()
test = elo_model_df[elo_model_df["date"] >= "2025-01-01"].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (19005, 28)
Test shape: (4319, 28)


Evaluate simple ELO accuracy using logistic regression

In [44]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

feature_cols = ["elo_diff"]

X_train = train[feature_cols]
y_train = train["A_won"]

X_test = test[feature_cols]
y_test = test["A_won"]

elo_logit = LogisticRegression()
elo_logit.fit(X_train, y_train)

pred_probs = elo_logit.predict_proba(X_test)[:, 1]
pred_class = (pred_probs >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, pred_class))
print("Log loss:", log_loss(y_test, pred_probs))
print("Brier score:", brier_score_loss(y_test, pred_probs))

Accuracy: 0.640194489465154
Log loss: 0.6297657230704258
Brier score: 0.22030783682409816


In [45]:
print("Intercept:", elo_logit.intercept_[0])
print("Elo coefficient:", elo_logit.coef_[0][0])

Intercept: 0.004899336708526259
Elo coefficient: 0.005137197106657147


Elo + Ranking Model

In [46]:
neutral_df["rank_diff"] = neutral_df["player_B_rank"] - neutral_df["player_A_rank"]

neutral_df["rank_points_diff"] = (
    neutral_df["player_A_rank_points"] - neutral_df["player_B_rank_points"]
)

neutral_df["log_rank_points_A"] = np.log1p(neutral_df["player_A_rank_points"])
neutral_df["log_rank_points_B"] = np.log1p(neutral_df["player_B_rank_points"])

neutral_df["log_rank_points_diff"] = (
    neutral_df["log_rank_points_A"] - neutral_df["log_rank_points_B"]
)

In [47]:
feature_cols = [
    "elo_diff",
    "rank_diff",
    "log_rank_points_diff"
]

rank_model_df = neutral_df.dropna(subset=feature_cols + ["A_won"]).copy()

train = rank_model_df[rank_model_df["date"] < "2025-01-01"].copy()
test = rank_model_df[rank_model_df["date"] >= "2025-01-01"].copy()

X_train = train[feature_cols]
y_train = train["A_won"]

X_test = test[feature_cols]
y_test = test["A_won"]

rank_logit = LogisticRegression(max_iter=1000)
rank_logit.fit(X_train, y_train)

pred_probs = rank_logit.predict_proba(X_test)[:, 1]
pred_class = (pred_probs >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, pred_class))
print("Log loss:", log_loss(y_test, pred_probs))
print("Brier score:", brier_score_loss(y_test, pred_probs))

Accuracy: 0.6467263306641545
Log loss: 0.6238291175118487
Brier score: 0.21753698471685465


In [48]:
# Same train/test rows as the Elo + ranking model
X_train_elo_same = train[["elo_diff"]]
X_test_elo_same = test[["elo_diff"]]

elo_logit_same = LogisticRegression(max_iter=1000)
elo_logit_same.fit(X_train_elo_same, y_train)

pred_probs_elo_same = elo_logit_same.predict_proba(X_test_elo_same)[:, 1]
pred_class_elo_same = (pred_probs_elo_same >= 0.5).astype(int)

print("Elo-only on same rows")
print("Accuracy:", accuracy_score(y_test, pred_class_elo_same))
print("Log loss:", log_loss(y_test, pred_probs_elo_same))
print("Brier score:", brier_score_loss(y_test, pred_probs_elo_same))

Elo-only on same rows
Accuracy: 0.641073951954781
Log loss: 0.6288842784494636
Brier score: 0.21992411441594245


Ranking does have genuine improvement of model, so we will continue with ELO difference, ranking difference, and rankining points difference as independent vars.

# Adding Surface ELO

In [49]:
def add_surface_elo_features(df, k=32, start_elo=1500):
    """
    Adds pre-match surface Elo ratings for winner and loser.
    Each player has a separate Elo rating for each surface.
    """
    df = df.copy()
    
    surface_ratings = {}
    
    w_surface_elo_pre = []
    l_surface_elo_pre = []
    
    for _, row in df.iterrows():
        winner = row["winner_id"]
        loser = row["loser_id"]
        surface = row["surface"]
        
        if pd.isna(surface):
            surface = "Unknown"
        
        winner_key = (winner, surface)
        loser_key = (loser, surface)
        
        winner_rating = surface_ratings.get(winner_key, start_elo)
        loser_rating = surface_ratings.get(loser_key, start_elo)
        
        w_surface_elo_pre.append(winner_rating)
        l_surface_elo_pre.append(loser_rating)
        
        p_winner = expected_win_prob(winner_rating, loser_rating)
        elo_change = k * (1 - p_winner)
        
        surface_ratings[winner_key] = winner_rating + elo_change
        surface_ratings[loser_key] = loser_rating - elo_change
    
    df["w_surface_elo_pre"] = w_surface_elo_pre
    df["l_surface_elo_pre"] = l_surface_elo_pre
    
    return df

In [50]:
match_df = add_surface_elo_features(match_df, k=32, start_elo=1500)
match_df[[
    "tourney_date",
    "surface",
    "winner_name",
    "loser_name",
    "w_surface_elo_pre",
    "l_surface_elo_pre"
]].tail(20)


,tourney_date,surface,winner_name,loser_name,w_surface_elo_pre,l_surface_elo_pre
23304,2026-06-11,Grass,Kamil Majchrzak,James McCabe,1537.201050,1482.762109
23305,2026-06-11,Grass,Felix Auger-Aliassime,Marton Fucsovics,1572.429767,1597.124280
23306,2026-06-12,Grass,Taylor Fritz,Martin Landaluce,1753.061846,1515.884348
23307,2026-06-12,Grass,Alexander Bublik,Giovanni Mpetshi Perricard,1684.983875,1526.257652
23308,2026-06-12,Grass,Ben Shelton,Marcos Giron,1546.812247,1580.178011
23309,2026-06-12,Grass,Taylor Fritz,Mattia Bellucci,1759.569996,1529.698152
23310,2026-06-12,Grass,Jiri Lehecka,Frances Tiafoe,1601.777382,1573.854551
23311,2026-06-12,Grass,Kamil Majchrzak,Felix Auger-Aliassime,1550.714361,1589.565080
23312,2026-06-12,Grass,Alex de Minaur,Benjamin Bonzi,1682.480884,1512.575321
23313,2026-06-12,Grass,Adrian Mannarino,Zhizhen Zhang,1597.233476,1535.891153


In [52]:
winner_is_A = neutral_df["A_won"].astype(bool).to_numpy()

neutral_df["player_A_surface_elo_pre"] = np.where(
    winner_is_A,
    match_df["w_surface_elo_pre"],
    match_df["l_surface_elo_pre"]
)

neutral_df["player_B_surface_elo_pre"] = np.where(
    winner_is_A,
    match_df["l_surface_elo_pre"],
    match_df["w_surface_elo_pre"]
)

neutral_df["surface_elo_diff"] = (
    neutral_df["player_A_surface_elo_pre"] 
    - neutral_df["player_B_surface_elo_pre"]
)

In [55]:
feature_cols = [
    "elo_diff",
    "surface_elo_diff",
    "rank_diff",
    "log_rank_points_diff"
]

surface_model_df = neutral_df.dropna(subset=feature_cols + ["A_won"]).copy()

train = surface_model_df[surface_model_df["date"] < "2025-01-01"].copy()
test = surface_model_df[surface_model_df["date"] >= "2025-01-01"].copy()

X_train = train[feature_cols]
y_train = train["A_won"]

X_test = test[feature_cols]
y_test = test["A_won"]

surface_logit = LogisticRegression(max_iter=1000)
surface_logit.fit(X_train, y_train)

pred_probs = surface_logit.predict_proba(X_test)[:, 1]
pred_class = (pred_probs >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, pred_class))
print("Log loss:", log_loss(y_test, pred_probs))
print("Brier score:", brier_score_loss(y_test, pred_probs))

Accuracy: 0.6467263306641545
Log loss: 0.6221735003435319
Brier score: 0.21680437216439483


How does model perform on grass only?

In [56]:
grass_test = test[test["surface"] == "Grass"].copy()

X_grass = grass_test[feature_cols]
y_grass = grass_test["A_won"]

grass_probs = surface_logit.predict_proba(X_grass)[:, 1]
grass_pred = (grass_probs >= 0.5).astype(int)

print("Grass test matches:", len(grass_test))
print("Grass accuracy:", accuracy_score(y_grass, grass_pred))
print("Grass log loss:", log_loss(y_grass, grass_probs))
print("Grass Brier score:", brier_score_loss(y_grass, grass_probs))

Grass test matches: 351
Grass accuracy: 0.6552706552706553
Grass log loss: 0.6259414164840589
Grass Brier score: 0.21827071704328982


In [57]:
rank_feature_cols = [
    "elo_diff",
    "rank_diff",
    "log_rank_points_diff"
]

X_train_rank = train[rank_feature_cols]
X_grass_rank = grass_test[rank_feature_cols]

rank_logit_grass_compare = LogisticRegression(max_iter=1000)
rank_logit_grass_compare.fit(X_train_rank, y_train)

grass_probs_rank = rank_logit_grass_compare.predict_proba(X_grass_rank)[:, 1]
grass_pred_rank = (grass_probs_rank >= 0.5).astype(int)

print("Elo + ranking on grass")
print("Accuracy:", accuracy_score(y_grass, grass_pred_rank))
print("Log loss:", log_loss(y_grass, grass_probs_rank))
print("Brier score:", brier_score_loss(y_grass, grass_probs_rank))

print("\nElo + ranking + surface Elo on grass")
print("Accuracy:", accuracy_score(y_grass, grass_pred))
print("Log loss:", log_loss(y_grass, grass_probs))
print("Brier score:", brier_score_loss(y_grass, grass_probs))

Elo + ranking on grass
Accuracy: 0.6723646723646723
Log loss: 0.6315920462335985
Brier score: 0.22018547741943875

Elo + ranking + surface Elo on grass
Accuracy: 0.6552706552706553
Log loss: 0.6259414164840589
Brier score: 0.21827071704328982


Recent Form

In [60]:
def add_rolling_form_features(df, windows=[10, 20]):
    """
    Adds pre-match rolling win percentage features for winner and loser.
    Uses only matches before the current match.
    """
    df = df.copy()
    
    # Overall match histories
    player_history = {}
    
    # Surface-specific histories
    player_surface_history = {}
    
    # Storage columns
    new_cols = {}
    
    for window in windows:
        new_cols[f"w_last{window}_win_pct_pre"] = []
        new_cols[f"l_last{window}_win_pct_pre"] = []
        new_cols[f"w_last{window}_matches_pre"] = []
        new_cols[f"l_last{window}_matches_pre"] = []
        
        new_cols[f"w_surface_last{window}_win_pct_pre"] = []
        new_cols[f"l_surface_last{window}_win_pct_pre"] = []
        new_cols[f"w_surface_last{window}_matches_pre"] = []
        new_cols[f"l_surface_last{window}_matches_pre"] = []
    
    for _, row in df.iterrows():
        winner = row["winner_id"]
        loser = row["loser_id"]
        surface = row["surface"]
        
        if pd.isna(surface):
            surface = "Unknown"
        
        for window in windows:
            # Overall histories before match
            w_hist = player_history.get(winner, [])
            l_hist = player_history.get(loser, [])
            
            w_recent = w_hist[-window:]
            l_recent = l_hist[-window:]
            
            # Use 0.5 if no history yet
            w_win_pct = np.mean(w_recent) if len(w_recent) > 0 else 0.5
            l_win_pct = np.mean(l_recent) if len(l_recent) > 0 else 0.5
            
            new_cols[f"w_last{window}_win_pct_pre"].append(w_win_pct)
            new_cols[f"l_last{window}_win_pct_pre"].append(l_win_pct)
            new_cols[f"w_last{window}_matches_pre"].append(len(w_recent))
            new_cols[f"l_last{window}_matches_pre"].append(len(l_recent))
            
            # Surface-specific histories before match
            w_surf_hist = player_surface_history.get((winner, surface), [])
            l_surf_hist = player_surface_history.get((loser, surface), [])
            
            w_surf_recent = w_surf_hist[-window:]
            l_surf_recent = l_surf_hist[-window:]
            
            w_surf_win_pct = np.mean(w_surf_recent) if len(w_surf_recent) > 0 else 0.5
            l_surf_win_pct = np.mean(l_surf_recent) if len(l_surf_recent) > 0 else 0.5
            
            new_cols[f"w_surface_last{window}_win_pct_pre"].append(w_surf_win_pct)
            new_cols[f"l_surface_last{window}_win_pct_pre"].append(l_surf_win_pct)
            new_cols[f"w_surface_last{window}_matches_pre"].append(len(w_surf_recent))
            new_cols[f"l_surface_last{window}_matches_pre"].append(len(l_surf_recent))
        
        # Update histories after recording pre-match features
        player_history.setdefault(winner, []).append(1)
        player_history.setdefault(loser, []).append(0)
        
        player_surface_history.setdefault((winner, surface), []).append(1)
        player_surface_history.setdefault((loser, surface), []).append(0)
    
    for col, values in new_cols.items():
        df[col] = values
    
    return df

In [61]:
match_df = add_rolling_form_features(match_df, windows=[10, 20])

In [64]:
winner_is_A = neutral_df["A_won"].astype(bool).to_numpy()

form_cols = [
    "last10_win_pct_pre",
    "last20_win_pct_pre",
    "last10_matches_pre",
    "last20_matches_pre",
    "surface_last10_win_pct_pre",
    "surface_last20_win_pct_pre",
    "surface_last10_matches_pre",
    "surface_last20_matches_pre"
]

for col in form_cols:
    neutral_df[f"player_A_{col}"] = np.where(
        winner_is_A,
        match_df[f"w_{col}"],
        match_df[f"l_{col}"]
    )
    
    neutral_df[f"player_B_{col}"] = np.where(
        winner_is_A,
        match_df[f"l_{col}"],
        match_df[f"w_{col}"]
    )
    
    neutral_df[f"{col}_diff"] = (
        neutral_df[f"player_A_{col}"] - neutral_df[f"player_B_{col}"]
    )

In [67]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

feature_cols = [
    "elo_diff",
    "surface_elo_diff",
    "rank_diff",
    "log_rank_points_diff",
    "last10_win_pct_pre_diff",
    "last20_win_pct_pre_diff",
    "surface_last10_win_pct_pre_diff",
    "surface_last20_win_pct_pre_diff",
    "last10_matches_pre_diff",
    "last20_matches_pre_diff",
    "surface_last10_matches_pre_diff",
    "surface_last20_matches_pre_diff"
]

form_model_df = neutral_df.dropna(subset=feature_cols + ["A_won"]).copy()

train = form_model_df[form_model_df["date"] < "2025-01-01"].copy()
test = form_model_df[form_model_df["date"] >= "2025-01-01"].copy()

X_train = train[feature_cols]
y_train = train["A_won"]

X_test = test[feature_cols]
y_test = test["A_won"]

form_logit_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=5000))
])

form_logit_scaled.fit(X_train, y_train)

pred_probs = form_logit_scaled.predict_proba(X_test)[:, 1]
pred_class = (pred_probs >= 0.5).astype(int)

print("Overall test")
print("Accuracy:", accuracy_score(y_test, pred_class))
print("Log loss:", log_loss(y_test, pred_probs))
print("Brier score:", brier_score_loss(y_test, pred_probs))

Overall test
Accuracy: 0.6490814884597268
Log loss: 0.6211988991476839
Brier score: 0.21653593906176977


In [68]:
grass_test = test[test["surface"] == "Grass"].copy()

X_grass = grass_test[feature_cols]
y_grass = grass_test["A_won"]

grass_probs = form_logit_scaled.predict_proba(X_grass)[:, 1]
grass_pred = (grass_probs >= 0.5).astype(int)

print("Grass test matches:", len(grass_test))
print("Grass accuracy:", accuracy_score(y_grass, grass_pred))
print("Grass log loss:", log_loss(y_grass, grass_probs))
print("Grass Brier score:", brier_score_loss(y_grass, grass_probs))

Grass test matches: 351
Grass accuracy: 0.6581196581196581
Grass log loss: 0.6221244064843009
Grass Brier score: 0.21669113179313498


In [69]:
logit_model = form_logit_scaled.named_steps["logit"]

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": logit_model.coef_[0]
}).sort_values("coefficient", ascending=False)

coef_df

,feature,coefficient
3,log_rank_points_diff,0.354975
1,surface_elo_diff,0.308741
0,elo_diff,0.293458
10,surface_last10_matches_pre_diff,0.095954
8,last10_matches_pre_diff,0.084657
4,last10_win_pct_pre_diff,0.078412
11,surface_last20_matches_pre_diff,0.032115
2,rank_diff,0.025492
6,surface_last10_win_pct_pre_diff,-0.002901
7,surface_last20_win_pct_pre_diff,-0.026223


This table suggests collinearity, but when tested model with less feauters, all important metrics worsened, so we should keep these features.

# Score Adjusted ELO

Instead of simply assigning each match an outcome 1 or 0 based on winner and loser, capture the competitiveness of the match using the score.

A 7-6, 7-6 win is very different from a 6-0, 6-2 win. 

We can use:

winner game pct = winner games/total games

Don't want a crushing victory or defeat to destroy the whole model though, so we should cap the effect of this. 

margin multiplier = 1 + 1.5 * (winner_game_pct - 0.6)
margin_multiplier = min(max(margin_multiplier, 0.75), 1.5)

In [72]:
import re
import numpy as np
import pandas as pd

def parse_tennis_score(score):
    """
    Parses ATP score string from the winner's perspective.
    
    Returns:
        winner_sets, loser_sets, winner_games, loser_games, valid_score
    """
    if pd.isna(score):
        return np.nan, np.nan, np.nan, np.nan, False
    
    score = str(score).strip()
    score_upper = score.upper()
    
    # For retirements/walkovers/defaults, do not use score margin.
    # We can still count the match for regular Elo, but margin should be neutral.
    bad_tokens = ["RET", "W/O", "DEF", "ABD", "DEF."]
    if any(token in score_upper for token in bad_tokens):
        return np.nan, np.nan, np.nan, np.nan, False
    
    # Remove tiebreak details, e.g. 7-6(5) -> 7-6
    clean_score = re.sub(r"\([^)]*\)", "", score)
    
    sets = clean_score.split()
    
    winner_sets = 0
    loser_sets = 0
    winner_games = 0
    loser_games = 0
    
    for s in sets:
        if "-" not in s:
            continue
        
        parts = s.split("-")
        if len(parts) < 2:
            continue
        
        try:
            w_games = int(parts[0])
            l_games = int(parts[1])
        except ValueError:
            continue
        
        winner_games += w_games
        loser_games += l_games
        
        if w_games > l_games:
            winner_sets += 1
        elif l_games > w_games:
            loser_sets += 1
    
    if winner_games + loser_games == 0:
        return np.nan, np.nan, np.nan, np.nan, False
    
    return winner_sets, loser_sets, winner_games, loser_games, True

In [75]:
score_parsed = match_df["score"].apply(parse_tennis_score)

match_df[[
    "winner_sets",
    "loser_sets",
    "winner_games",
    "loser_games",
    "valid_score"
]] = pd.DataFrame(score_parsed.tolist(), index=match_df.index)




In [76]:
match_df["winner_game_pct"] = (
    match_df["winner_games"] / 
    (match_df["winner_games"] + match_df["loser_games"])
)

match_df["game_diff"] = match_df["winner_games"] - match_df["loser_games"]
match_df["set_diff"] = match_df["winner_sets"] - match_df["loser_sets"]

In [79]:
def score_margin_multiplier(winner_game_pct):
    """
    Converts winner game percentage into an Elo update multiplier.
    
    60% of games won is treated as a normal win.
    Close wins get smaller updates.
    Dominant wins get larger updates.
    """
    if pd.isna(winner_game_pct):
        return 1.0
    
    mult = 1 + 1.5 * (winner_game_pct - 0.60)
    
    # Cap it so we don't overreact to one blowout
    mult = max(0.75, min(1.50, mult))
    
    return mult

match_df["score_margin_mult"] = match_df["winner_game_pct"].apply(score_margin_multiplier)

In [82]:
def add_score_adjusted_elo_features(df, k=32, start_elo=1500):
    """
    Adds score-adjusted pre-match Elo and score-adjusted pre-match surface Elo.
    
    The score margin only affects the update AFTER the current match,
    so the pre-match features are still leakage-safe.
    """
    df = df.copy()
    
    overall_ratings = {}
    surface_ratings = {}
    
    w_score_elo_pre = []
    l_score_elo_pre = []
    w_score_surface_elo_pre = []
    l_score_surface_elo_pre = []
    
    for _, row in df.iterrows():
        winner = row["winner_id"]
        loser = row["loser_id"]
        surface = row["surface"]
        
        if pd.isna(surface):
            surface = "Unknown"
        
        margin_mult = row.get("score_margin_mult", 1.0)
        if pd.isna(margin_mult):
            margin_mult = 1.0
        
        # Overall score-adjusted Elo before match
        w_rating = overall_ratings.get(winner, start_elo)
        l_rating = overall_ratings.get(loser, start_elo)
        
        w_score_elo_pre.append(w_rating)
        l_score_elo_pre.append(l_rating)
        
        p_winner = expected_win_prob(w_rating, l_rating)
        elo_change = k * margin_mult * (1 - p_winner)
        
        overall_ratings[winner] = w_rating + elo_change
        overall_ratings[loser] = l_rating - elo_change
        
        # Surface score-adjusted Elo before match
        w_key = (winner, surface)
        l_key = (loser, surface)
        
        w_surface_rating = surface_ratings.get(w_key, start_elo)
        l_surface_rating = surface_ratings.get(l_key, start_elo)
        
        w_score_surface_elo_pre.append(w_surface_rating)
        l_score_surface_elo_pre.append(l_surface_rating)
        
        p_winner_surface = expected_win_prob(w_surface_rating, l_surface_rating)
        surface_elo_change = k * margin_mult * (1 - p_winner_surface)
        
        surface_ratings[w_key] = w_surface_rating + surface_elo_change
        surface_ratings[l_key] = l_surface_rating - surface_elo_change
    
    df["w_score_elo_pre"] = w_score_elo_pre
    df["l_score_elo_pre"] = l_score_elo_pre
    df["w_score_surface_elo_pre"] = w_score_surface_elo_pre
    df["l_score_surface_elo_pre"] = l_score_surface_elo_pre
    
    return df

match_df = add_score_adjusted_elo_features(match_df, k=32, start_elo=1500)

In [83]:
match_df[[
    "w_score_elo_pre",
    "l_score_elo_pre",
    "w_score_surface_elo_pre",
    "l_score_surface_elo_pre"
]].describe()

,w_score_elo_pre,l_score_elo_pre,w_score_surface_elo_pre,l_score_surface_elo_pre
count,23324.000000,23324.000000,23324.000000,23324.000000
mean,1643.791192,1587.382348,1605.142904,1559.524584
std,157.490325,121.889294,134.635147,103.340769
min,1313.758333,1322.089677,1284.376090,1260.534114
25%,1521.917942,1500.000000,1503.992258,1493.794545
50%,1609.158255,1559.780204,1568.636890,1530.645407
75%,1731.671305,1650.749480,1673.738044,1605.344341
max,2292.940689,2293.297749,2219.522294,2222.227184


In [85]:
winner_is_A = neutral_df["A_won"].astype(bool).to_numpy()

neutral_df["player_A_score_elo_pre"] = np.where(
    winner_is_A,
    match_df["w_score_elo_pre"],
    match_df["l_score_elo_pre"]
)

neutral_df["player_B_score_elo_pre"] = np.where(
    winner_is_A,
    match_df["l_score_elo_pre"],
    match_df["w_score_elo_pre"]
)

neutral_df["score_elo_diff"] = (
    neutral_df["player_A_score_elo_pre"] - neutral_df["player_B_score_elo_pre"]
)

neutral_df["player_A_score_surface_elo_pre"] = np.where(
    winner_is_A,
    match_df["w_score_surface_elo_pre"],
    match_df["l_score_surface_elo_pre"]
)

neutral_df["player_B_score_surface_elo_pre"] = np.where(
    winner_is_A,
    match_df["l_score_surface_elo_pre"],
    match_df["w_score_surface_elo_pre"]
)

neutral_df["score_surface_elo_diff"] = (
    neutral_df["player_A_score_surface_elo_pre"] 
    - neutral_df["player_B_score_surface_elo_pre"]
)

In [89]:
score_feature_cols = [
    "score_elo_diff",
    "score_surface_elo_diff",
    "rank_diff",
    "log_rank_points_diff",
    "last10_win_pct_pre_diff",
    "last20_win_pct_pre_diff",
    "surface_last10_win_pct_pre_diff",
    "surface_last20_win_pct_pre_diff",
    "last10_matches_pre_diff",
    "last20_matches_pre_diff",
    "surface_last10_matches_pre_diff",
    "surface_last20_matches_pre_diff"
]

score_model_df = neutral_df.dropna(subset=score_feature_cols + ["A_won"]).copy()

train = score_model_df[score_model_df["date"] < "2025-01-01"].copy()
test = score_model_df[score_model_df["date"] >= "2025-01-01"].copy()

X_train = train[score_feature_cols]
y_train = train["A_won"]

X_test = test[score_feature_cols]
y_test = test["A_won"]

score_logit_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=5000))
])

score_logit_scaled.fit(X_train, y_train)

pred_probs = score_logit_scaled.predict_proba(X_test)[:, 1]
pred_class = (pred_probs >= 0.5).astype(int)

print("Overall test")
print("Accuracy:", accuracy_score(y_test, pred_class))
print("Log loss:", log_loss(y_test, pred_probs))
print("Brier score:", brier_score_loss(y_test, pred_probs))

grass_test = test[test["surface"] == "Grass"].copy()

X_grass = grass_test[score_feature_cols]
y_grass = grass_test["A_won"]

grass_probs = score_logit_scaled.predict_proba(X_grass)[:, 1]
grass_pred = (grass_probs >= 0.5).astype(int)

print("Grass test matches:", len(grass_test))
print("Grass accuracy:", accuracy_score(y_grass, grass_pred))
print("Grass log loss:", log_loss(y_grass, grass_probs))
print("Grass Brier score:", brier_score_loss(y_grass, grass_probs))

Overall test
Accuracy: 0.6507300989166274
Log loss: 0.6209224876498555
Brier score: 0.2164089997197555
Grass test matches: 351
Grass accuracy: 0.6581196581196581
Grass log loss: 0.6207876867501573
Grass Brier score: 0.21615898172723466


In [90]:
logit_model = score_logit_scaled.named_steps["logit"]

score_coef_df = pd.DataFrame({
    "feature": score_feature_cols,
    "coefficient": logit_model.coef_[0]
}).sort_values("coefficient", ascending=False)

score_coef_df

,feature,coefficient
0,score_elo_diff,0.327146
1,score_surface_elo_diff,0.319875
3,log_rank_points_diff,0.312022
10,surface_last10_matches_pre_diff,0.100586
8,last10_matches_pre_diff,0.085014
4,last10_win_pct_pre_diff,0.075977
2,rank_diff,0.049346
11,surface_last20_matches_pre_diff,0.025024
6,surface_last10_win_pct_pre_diff,-0.000477
7,surface_last20_win_pct_pre_diff,-0.032037


Running score-adjusted and regular elo together

In [92]:
both_elo_feature_cols = [
    "elo_diff",
    "surface_elo_diff",
    "score_elo_diff",
    "score_surface_elo_diff",
    "rank_diff",
    "log_rank_points_diff",
    "last10_win_pct_pre_diff",
    "last20_win_pct_pre_diff",
    "surface_last10_win_pct_pre_diff",
    "surface_last20_win_pct_pre_diff",
    "last10_matches_pre_diff",
    "last20_matches_pre_diff",
    "surface_last10_matches_pre_diff",
    "surface_last20_matches_pre_diff"
]

both_model_df = neutral_df.dropna(subset=both_elo_feature_cols + ["A_won"]).copy()

train = both_model_df[both_model_df["date"] < "2025-01-01"].copy()
test = both_model_df[both_model_df["date"] >= "2025-01-01"].copy()

X_train = train[both_elo_feature_cols]
y_train = train["A_won"]

X_test = test[both_elo_feature_cols]
y_test = test["A_won"]

both_logit_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=5000))
])

both_logit_scaled.fit(X_train, y_train)

pred_probs = both_logit_scaled.predict_proba(X_test)[:, 1]
pred_class = (pred_probs >= 0.5).astype(int)

print("Overall test")
print("Accuracy:", accuracy_score(y_test, pred_class))
print("Log loss:", log_loss(y_test, pred_probs))
print("Brier score:", brier_score_loss(y_test, pred_probs))

grass_test = test[test["surface"] == "Grass"].copy()

X_grass = grass_test[both_elo_feature_cols]
y_grass = grass_test["A_won"]

grass_probs = both_logit_scaled.predict_proba(X_grass)[:, 1]
grass_pred = (grass_probs >= 0.5).astype(int)

print("Grass test matches:", len(grass_test))
print("Grass accuracy:", accuracy_score(y_grass, grass_pred))
print("Grass log loss:", log_loss(y_grass, grass_probs))
print("Grass Brier score:", brier_score_loss(y_grass, grass_probs))

Overall test
Accuracy: 0.6570890249646726
Log loss: 0.6209842290849419
Brier score: 0.21643350132359895
Grass test matches: 351
Grass accuracy: 0.6609686609686609
Grass log loss: 0.6195281002237595
Grass Brier score: 0.21570966314853085


In [93]:
both_logit_model = both_logit_scaled.named_steps["logit"]

both_coef_df = pd.DataFrame({
    "feature": both_elo_feature_cols,
    "coefficient": both_logit_model.coef_[0]
}).sort_values("coefficient", ascending=False)

both_coef_df

,feature,coefficient
2,score_elo_diff,1.337085
3,score_surface_elo_diff,1.331573
5,log_rank_points_diff,0.319961
12,surface_last10_matches_pre_diff,0.095964
6,last10_win_pct_pre_diff,0.087818
10,last10_matches_pre_diff,0.086703
4,rank_diff,0.045393
13,surface_last20_matches_pre_diff,0.030518
8,surface_last10_win_pct_pre_diff,0.005369
9,surface_last20_win_pct_pre_diff,-0.028770


In [94]:
elo_corr_cols = [
    "elo_diff",
    "surface_elo_diff",
    "score_elo_diff",
    "score_surface_elo_diff"
]

neutral_df[elo_corr_cols].corr()

,elo_diff,surface_elo_diff,score_elo_diff,score_surface_elo_diff
elo_diff,1.000000,0.847958,0.998422,0.849392
surface_elo_diff,0.847958,1.000000,0.848411,0.998147
score_elo_diff,0.998422,0.848411,1.000000,0.852315
score_surface_elo_diff,0.849392,0.998147,0.852315,1.000000


### Multicollinearity Check: Regular Elo vs Score-Adjusted Elo

Regular Elo and score-adjusted Elo are extremely highly correlated:

- `elo_diff` vs `score_elo_diff`: ~0.998
- `surface_elo_diff` vs `score_surface_elo_diff`: ~0.998

Because of this, the model with both regular and score-adjusted Elo has unstable and hard-to-interpret coefficients. The both-Elo model slightly improved grass metrics, but the score-adjusted-only model is cleaner and more interpretable.

For now, use the score-adjusted-only feature set as the main model, while keeping the both-Elo model as a candidate for later Wimbledon-specific backtesting.

### Head to head feature

In [ ]:
def add_h2h_features(df):
    """
    Adds pre-match head-to-head features.
    
    For each match:
    - records the winner's previous record vs the loser
    - records the loser's previous record vs the winner
    - then updates the H2H record after the match
    
    This is leakage-safe because current match is not included in pre-match features.
    """
    
    df = df.copy()
    
    # ordered pair: (player_id, opponent_id) -> wins by player_id over opponent_id
    h2h_wins = {}
    
    w_h2h_wins_pre = []
    w_h2h_losses_pre = []
    w_h2h_matches_pre = []
    w_h2h_win_pct_pre = []
    
    l_h2h_wins_pre = []
    l_h2h_losses_pre = []
    l_h2h_matches_pre = []
    l_h2h_win_pct_pre = []
    
    for _, row in df.iterrows():
        winner = row["winner_id"]
        loser = row["loser_id"]
        
        # Winner's previous wins/losses against this loser
        w_prev_wins = h2h_wins.get((winner, loser), 0)
        w_prev_losses = h2h_wins.get((loser, winner), 0)
        w_prev_matches = w_prev_wins + w_prev_losses
        
        # Loser's previous wins/losses against this winner
        l_prev_wins = w_prev_losses
        l_prev_losses = w_prev_wins
        l_prev_matches = w_prev_matches
        
        # Use 0.5 when they have never played before
        if w_prev_matches > 0:
            w_prev_pct = w_prev_wins / w_prev_matches
            l_prev_pct = l_prev_wins / l_prev_matches
        else:
            w_prev_pct = 0.5
            l_prev_pct = 0.5
        
        # Store pre-match features
        w_h2h_wins_pre.append(w_prev_wins)
        w_h2h_losses_pre.append(w_prev_losses)
        w_h2h_matches_pre.append(w_prev_matches)
        w_h2h_win_pct_pre.append(w_prev_pct)
        
        l_h2h_wins_pre.append(l_prev_wins)
        l_h2h_losses_pre.append(l_prev_losses)
        l_h2h_matches_pre.append(l_prev_matches)
        l_h2h_win_pct_pre.append(l_prev_pct)
        
        # Update H2H after the match
        h2h_wins[(winner, loser)] = h2h_wins.get((winner, loser), 0) + 1
    
    df["w_h2h_wins_pre"] = w_h2h_wins_pre
    df["w_h2h_losses_pre"] = w_h2h_losses_pre
    df["w_h2h_matches_pre"] = w_h2h_matches_pre
    df["w_h2h_win_pct_pre"] = w_h2h_win_pct_pre
    
    df["l_h2h_wins_pre"] = l_h2h_wins_pre
    df["l_h2h_losses_pre"] = l_h2h_losses_pre
    df["l_h2h_matches_pre"] = l_h2h_matches_pre
    df["l_h2h_win_pct_pre"] = l_h2h_win_pct_pre
    
    return df

match_df = add_h2h_features(match_df)

In [145]:
match_df["w_h2h_advantage_pre"] = (
    match_df["w_h2h_win_pct_pre"] - 0.5
)

match_df["l_h2h_advantage_pre"] = (
    match_df["l_h2h_win_pct_pre"] - 0.5
)

match_df["w_h2h_weighted_advantage_pre"] = (
    match_df["w_h2h_advantage_pre"] * np.log1p(match_df["w_h2h_matches_pre"])
)

match_df["l_h2h_weighted_advantage_pre"] = (
    match_df["l_h2h_advantage_pre"] * np.log1p(match_df["l_h2h_matches_pre"])
)

In [146]:
winner_is_A = neutral_df["A_won"].astype(bool).to_numpy()

neutral_df["player_A_h2h_win_pct_pre"] = np.where(
    winner_is_A,
    match_df["w_h2h_win_pct_pre"],
    match_df["l_h2h_win_pct_pre"]
)

neutral_df["player_B_h2h_win_pct_pre"] = np.where(
    winner_is_A,
    match_df["l_h2h_win_pct_pre"],
    match_df["w_h2h_win_pct_pre"]
)

neutral_df["h2h_win_pct_diff"] = (
    neutral_df["player_A_h2h_win_pct_pre"] 
    - neutral_df["player_B_h2h_win_pct_pre"]
)

neutral_df["h2h_matches_pre"] = np.where(
    winner_is_A,
    match_df["w_h2h_matches_pre"],
    match_df["l_h2h_matches_pre"]
)

neutral_df["player_A_h2h_weighted_advantage_pre"] = np.where(
    winner_is_A,
    match_df["w_h2h_weighted_advantage_pre"],
    match_df["l_h2h_weighted_advantage_pre"]
)

neutral_df["player_B_h2h_weighted_advantage_pre"] = np.where(
    winner_is_A,
    match_df["l_h2h_weighted_advantage_pre"],
    match_df["w_h2h_weighted_advantage_pre"]
)

neutral_df["h2h_weighted_advantage_diff"] = (
    neutral_df["player_A_h2h_weighted_advantage_pre"]
    - neutral_df["player_B_h2h_weighted_advantage_pre"]
)

In [147]:
neutral_df[[
    "player_A_name",
    "player_B_name",
    "A_won",
    "h2h_matches_pre",
    "player_A_h2h_win_pct_pre",
    "player_B_h2h_win_pct_pre",
    "h2h_win_pct_diff",
    "h2h_weighted_advantage_diff"
]].tail(20)

,player_A_name,player_B_name,A_won,h2h_matches_pre,player_A_h2h_win_pct_pre,player_B_h2h_win_pct_pre,h2h_win_pct_diff,h2h_weighted_advantage_diff
23304,James McCabe,Kamil Majchrzak,0,0,0.500000,0.500000,0.000000,0.000000
23305,Felix Auger-Aliassime,Marton Fucsovics,1,6,0.666667,0.333333,0.333333,0.648637
23306,Martin Landaluce,Taylor Fritz,0,0,0.500000,0.500000,0.000000,0.000000
23307,Alexander Bublik,Giovanni Mpetshi Perricard,1,3,0.333333,0.666667,-0.333333,-0.462098
23308,Ben Shelton,Marcos Giron,1,3,0.666667,0.333333,0.333333,0.462098
23309,Mattia Bellucci,Taylor Fritz,0,0,0.500000,0.500000,0.000000,0.000000
23310,Jiri Lehecka,Frances Tiafoe,1,3,0.333333,0.666667,-0.333333,-0.462098
23311,Felix Auger-Aliassime,Kamil Majchrzak,0,0,0.500000,0.500000,0.000000,0.000000
23312,Alex de Minaur,Benjamin Bonzi,1,4,0.750000,0.250000,0.500000,0.804719
23313,Adrian Mannarino,Zhizhen Zhang,1,2,0.500000,0.500000,0.000000,0.000000


# Backtesting

In [165]:
wimbledon_rows = neutral_df[
    neutral_df["tourney_name"].str.contains("Wimbledon", case=False, na=False)
].copy()

wimbledon_rows[["date", "tourney_name", "surface", "round", "player_A_name", "player_B_name", "A_won"]].head()

score_feature_cols = [
    "score_elo_diff",
    "score_surface_elo_diff",
    "rank_diff",
    "log_rank_points_diff",
    "last10_win_pct_pre_diff",
    "last20_win_pct_pre_diff",
    "surface_last10_win_pct_pre_diff",
    "surface_last20_win_pct_pre_diff",
    "last10_matches_pre_diff",
    "last20_matches_pre_diff",
    "surface_last10_matches_pre_diff",
    "surface_last20_matches_pre_diff"
]

In [166]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss
import pandas as pd
import numpy as np

def backtest_wimbledon_model(df, feature_cols, years=None, model_name="model"):
    """
    For each Wimbledon year:
    - train on all matches before that year's Wimbledon
    - test on that year's Wimbledon matches
    """
    
    results = []
    all_preds = []
    
    wim_all = df[
        df["tourney_name"].str.contains("Wimbledon", case=False, na=False)
    ].copy()
    
    if years is None:
        years = sorted(wim_all["date"].dt.year.unique())
    
    for year in years:
        wim_year = wim_all[wim_all["date"].dt.year == year].copy()
        
        if len(wim_year) == 0:
            continue
        
        # Wimbledon start date for that year
        cutoff_date = wim_year["date"].min()
        
        # Train only on matches before Wimbledon started
        train = df[df["date"] < cutoff_date].copy()
        test = wim_year.copy()
        
        # Drop rows with missing model features
        train = train.dropna(subset=feature_cols + ["A_won"])
        test = test.dropna(subset=feature_cols + ["A_won"])
        
        if len(train) == 0 or len(test) == 0:
            continue
        
        X_train = train[feature_cols]
        y_train = train["A_won"]
        
        X_test = test[feature_cols]
        y_test = test["A_won"]
        
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(max_iter=5000))
        ])
        
        model.fit(X_train, y_train)
        
        pred_probs = model.predict_proba(X_test)[:, 1]
        pred_class = (pred_probs >= 0.5).astype(int)
        
        year_result = {
            "model": model_name,
            "year": year,
            "wimbledon_start": cutoff_date,
            "train_matches": len(train),
            "test_matches": len(test),
            "accuracy": accuracy_score(y_test, pred_class),
            "log_loss": log_loss(y_test, pred_probs),
            "brier": brier_score_loss(y_test, pred_probs)
        }
        
        results.append(year_result)
        
        pred_df = test.copy()
        pred_df["model"] = model_name
        pred_df["pred_prob_A_wins"] = pred_probs
        pred_df["pred_class_A_wins"] = pred_class
        pred_df["correct"] = pred_df["pred_class_A_wins"] == pred_df["A_won"]
        pred_df["year"] = year
        
        all_preds.append(pred_df)
    
    results_df = pd.DataFrame(results)
    
    if len(all_preds) > 0:
        preds_df = pd.concat(all_preds, ignore_index=True)
    else:
        preds_df = pd.DataFrame()
    
    return results_df, preds_df

In [167]:
wim_years = [2019, 2021, 2022, 2023, 2024, 2025]

score_wim_results, score_wim_preds = backtest_wimbledon_model(
    neutral_df,
    score_feature_cols,
    years=wim_years,
    model_name="score_adjusted_elo_model"
)

print(score_wim_results)
print(score_wim_results[["accuracy", "log_loss", "brier"]].mean())
print(score_wim_results[["year", "test_matches", "accuracy", "log_loss", "brier"]])

                      model  year wimbledon_start  train_matches  \
0  score_adjusted_elo_model  2019      2019-07-01           4414   
1  score_adjusted_elo_model  2021      2021-06-28           8506   
2  score_adjusted_elo_model  2022      2022-06-27          11363   
3  score_adjusted_elo_model  2023      2023-07-03          14317   
4  score_adjusted_elo_model  2024      2024-07-01          17287   
5  score_adjusted_elo_model  2025      2025-06-30          20224   

   test_matches  accuracy  log_loss     brier  
0           127  0.724409  0.559918  0.191170  
1           127  0.716535  0.545673  0.182914  
2           127  0.724409  0.539461  0.180067  
3           127  0.716535  0.551915  0.186530  
4           127  0.740157  0.523848  0.174255  
5           127  0.677165  0.593498  0.204267  
accuracy    0.716535
log_loss    0.552386
brier       0.186534
dtype: float64
   year  test_matches  accuracy  log_loss     brier
0  2019           127  0.724409  0.559918  0.191170
1  20

In [168]:
both_elo_feature_cols = [
    "elo_diff",
    "surface_elo_diff",
    "score_elo_diff",
    "score_surface_elo_diff",
    "rank_diff",
    "log_rank_points_diff",
    "last10_win_pct_pre_diff",
    "last20_win_pct_pre_diff",
    "surface_last10_win_pct_pre_diff",
    "surface_last20_win_pct_pre_diff",
    "last10_matches_pre_diff",
    "last20_matches_pre_diff",
    "surface_last10_matches_pre_diff",
    "surface_last20_matches_pre_diff"
]

both_wim_results, both_wim_preds = backtest_wimbledon_model(
    neutral_df,
    both_elo_feature_cols,
    years=wim_years,
    model_name="both_elo_model"
)

both_wim_results

,model,year,wimbledon_start,train_matches,test_matches,accuracy,log_loss,brier
0,both_elo_model,2019,2019-07-01,4414,127,0.732283,0.548648,0.186050
1,both_elo_model,2021,2021-06-28,8506,127,0.748031,0.544888,0.182469
2,both_elo_model,2022,2022-06-27,11363,127,0.763780,0.543901,0.181750
3,both_elo_model,2023,2023-07-03,14317,127,0.724409,0.553670,0.187376
4,both_elo_model,2024,2024-07-01,17287,127,0.763780,0.516552,0.170484
5,both_elo_model,2025,2025-06-30,20224,127,0.692913,0.588386,0.202232


In [169]:
comparison = pd.concat([score_wim_results, both_wim_results], ignore_index=True)

comparison.groupby("model").agg(
    avg_accuracy=("accuracy", "mean"),
    avg_log_loss=("log_loss", "mean"),
    avg_brier=("brier", "mean"),
    total_test_matches=("test_matches", "sum")
).reset_index()

,model,avg_accuracy,avg_log_loss,avg_brier,total_test_matches
0,both_elo_model,0.737533,0.549341,0.185060,762
1,score_adjusted_elo_model,0.716535,0.552386,0.186534,762


Although regular Elo and score-adjusted Elo are highly correlated, the model using both versions performed best in Wimbledon-specific backtesting. Because the goal is predictive bracket performance rather than coefficient interpretation, we will keep the both-Elo model as the current main model. Coefficients from this model should not be interpreted causally because of multicollinearity.

In [170]:
round_order = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7
}

round_perf = both_wim_preds.groupby("round").agg(
    matches=("A_won", "size"),
    accuracy=("correct", "mean"),
    avg_pred_prob_A=("pred_prob_A_wins", "mean"),
    actual_A_win_rate=("A_won", "mean")
).reset_index()

round_perf["round_order"] = round_perf["round"].map(round_order)
round_perf = round_perf.sort_values("round_order")

round_perf

,round,matches,accuracy,avg_pred_prob_A,actual_A_win_rate,round_order
2,R128,384,0.708333,0.493613,0.484375,1
5,R64,192,0.739583,0.519586,0.552083,2
4,R32,96,0.781250,0.513738,0.479167,3
3,R16,48,0.791667,0.435261,0.458333,4
1,QF,24,0.875000,0.375637,0.333333,5
6,SF,12,0.833333,0.499836,0.333333,6
0,F,6,0.666667,0.537027,0.500000,7


In [171]:
both_wim_preds["predicted_winner_prob"] = np.where(
    both_wim_preds["pred_class_A_wins"] == 1,
    both_wim_preds["pred_prob_A_wins"],
    1 - both_wim_preds["pred_prob_A_wins"]
)

round_metrics = []

for rnd, group in both_wim_preds.groupby("round"):
    round_metrics.append({
        "round": rnd,
        "matches": len(group),
        "accuracy": group["correct"].mean(),
        "avg_predicted_winner_prob": group["predicted_winner_prob"].mean(),
        "brier": brier_score_loss(group["A_won"], group["pred_prob_A_wins"]),
        "log_loss": log_loss(group["A_won"], group["pred_prob_A_wins"])
    })

round_metrics_df = pd.DataFrame(round_metrics)

round_order = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7
}

round_metrics_df["round_order"] = round_metrics_df["round"].map(round_order)
round_metrics_df = round_metrics_df.sort_values("round_order")

round_metrics_df

,round,matches,accuracy,avg_predicted_winner_prob,brier,log_loss,round_order
2,R128,384,0.708333,0.667142,0.197782,0.578982,1
5,R64,192,0.739583,0.681978,0.188096,0.554595,2
4,R32,96,0.781250,0.685216,0.156502,0.483653,3
3,R16,48,0.791667,0.709654,0.161930,0.495905,4
1,QF,24,0.875000,0.734133,0.123610,0.413134,5
6,SF,12,0.833333,0.700906,0.156289,0.492659,6
0,F,6,0.666667,0.641227,0.219049,0.620875,7


In [172]:
def validate_bracket_order(match_df, year):
    wim = match_df[
        (match_df["tourney_name"].str.contains("Wimbledon", case=False, na=False)) &
        (match_df["tourney_date"].dt.year == year)
    ].copy()
    
    round_sequence = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]
    validation_rows = []
    
    for i in range(len(round_sequence) - 1):
        prev_round = round_sequence[i]
        next_round = round_sequence[i + 1]
        
        prev_matches = wim[wim["round"] == prev_round].sort_values("match_num").copy()
        next_matches = wim[wim["round"] == next_round].sort_values("match_num").copy()
        
        prev_winners = prev_matches["winner_id"].tolist()
        
        expected_pairs = [
            set(prev_winners[j:j+2]) 
            for j in range(0, len(prev_winners), 2)
        ]
        
        actual_pairs = [
            set([row["winner_id"], row["loser_id"]])
            for _, row in next_matches.iterrows()
        ]
        
        num_mismatches = sum(
            expected != actual
            for expected, actual in zip(expected_pairs, actual_pairs)
        )
        
        validation_rows.append({
            "year": year,
            "from_round": prev_round,
            "to_round": next_round,
            "expected_matches": len(expected_pairs),
            "actual_matches": len(actual_pairs),
            "mismatches": num_mismatches
        })
    
    return pd.DataFrame(validation_rows)

In [173]:
for year in [2019, 2021, 2022, 2023, 2024, 2025]:
    print("\nYEAR:", year)
    display(validate_bracket_order(match_df, year))


YEAR: 2019


,year,from_round,to_round,expected_matches,actual_matches,mismatches
0,2019,R128,R64,32,32,32
1,2019,R64,R32,16,16,16
2,2019,R32,R16,8,8,8
3,2019,R16,QF,4,4,4
4,2019,QF,SF,2,2,2
5,2019,SF,F,1,1,0



YEAR: 2021


,year,from_round,to_round,expected_matches,actual_matches,mismatches
0,2021,R128,R64,32,32,32
1,2021,R64,R32,16,16,16
2,2021,R32,R16,8,8,8
3,2021,R16,QF,4,4,3
4,2021,QF,SF,2,2,2
5,2021,SF,F,1,1,0



YEAR: 2022


...

,year,from_round,to_round,expected_matches,actual_matches,mismatches
0,2023,R128,R64,32,32,0
1,2023,R64,R32,16,16,0
2,2023,R32,R16,8,8,0
3,2023,R16,QF,4,4,0
4,2023,QF,SF,2,2,0
5,2023,SF,F,1,1,0



YEAR: 2024


,year,from_round,to_round,expected_matches,actual_matches,mismatches
0,2024,R128,R64,32,32,32
1,2024,R64,R32,16,16,15
2,2024,R32,R16,8,8,8
3,2024,R16,QF,4,4,4
4,2024,QF,SF,2,2,0
5,2024,SF,F,1,1,0



YEAR: 2025


,year,from_round,to_round,expected_matches,actual_matches,mismatches
0,2025,R128,R64,32,32,32
1,2025,R64,R32,16,16,16
2,2025,R32,R16,8,8,8
3,2025,R16,QF,4,4,4
4,2025,QF,SF,2,2,0
5,2025,SF,F,1,1,0


Cannot use match number to make brackets, need to make bracket tree manually

In [174]:
ROUND_SEQUENCE = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]

def build_wimbledon_bracket_tree(match_df, year):
    """
    Reconstructs the Wimbledon bracket tree using actual round-to-round player progression.
    
    Each match becomes a node.
    Later-round nodes point to the two earlier-round nodes that fed into it.
    """
    wim = match_df[
        (match_df["tourney_name"].str.contains("Wimbledon", case=False, na=False)) &
        (match_df["tourney_date"].dt.year == year)
    ].copy()
    
    # Create match nodes by round
    nodes_by_round = {}
    
    for rnd in ROUND_SEQUENCE:
        round_matches = wim[wim["round"] == rnd].copy().reset_index(drop=True)
        
        nodes = []
        for idx, row in round_matches.iterrows():
            node = {
                "round": rnd,
                "round_index": idx,
                "winner_id": row["winner_id"],
                "winner_name": row["winner_name"],
                "loser_id": row["loser_id"],
                "loser_name": row["loser_name"],
                "player_ids": {row["winner_id"], row["loser_id"]},
                "left": None,
                "right": None,
                "row": row
            }
            nodes.append(node)
        
        nodes_by_round[rnd] = nodes
    
    # Link each later-round match to the two previous-round matches
    for i in range(1, len(ROUND_SEQUENCE)):
        prev_round = ROUND_SEQUENCE[i - 1]
        curr_round = ROUND_SEQUENCE[i]
        
        prev_nodes = nodes_by_round[prev_round]
        curr_nodes = nodes_by_round[curr_round]
        
        for curr_node in curr_nodes:
            feeders = []
            
            for prev_node in prev_nodes:
                # The previous-round winner must appear in the current match
                if prev_node["winner_id"] in curr_node["player_ids"]:
                    feeders.append(prev_node)
            
            if len(feeders) != 2:
                print(
                    f"WARNING: {year} {curr_round} match {curr_node['round_index']} "
                    f"has {len(feeders)} feeders instead of 2"
                )
            
            if len(feeders) >= 1:
                curr_node["left"] = feeders[0]
            if len(feeders) >= 2:
                curr_node["right"] = feeders[1]
    
    # Final node is the root of the bracket tree
    final_nodes = nodes_by_round["F"]
    
    if len(final_nodes) != 1:
        raise ValueError(f"Expected exactly one final node for {year}, found {len(final_nodes)}")
    
    return final_nodes[0], nodes_by_round

In [175]:
def count_leaf_matches(node):
    """
    Counts R128 matches underneath a node.
    """
    if node["round"] == "R128":
        return 1
    
    return count_leaf_matches(node["left"]) + count_leaf_matches(node["right"])


for year in [2019, 2021, 2022, 2023, 2024, 2025]:
    root, nodes_by_round = build_wimbledon_bracket_tree(match_df, year)
    
    print("\nYEAR:", year)
    print("Final:", root["winner_name"], "def.", root["loser_name"])
    print("R128 leaf matches under final tree:", count_leaf_matches(root))
    
    for rnd in ROUND_SEQUENCE:
        print(rnd, len(nodes_by_round[rnd]))


YEAR: 2019
Final: Novak Djokovic def. Roger Federer
R128 leaf matches under final tree: 64
R128 64
R64 32
R32 16
R16 8
QF 4
SF 2
F 1

YEAR: 2021
Final: Novak Djokovic def. Matteo Berrettini
R128 leaf matches under final tree: 64
R128 64
R64 32
R32 16
R16 8
QF 4
SF 2
F 1

YEAR: 2022
Final: Novak Djokovic def. Nick Kyrgios
R128 leaf matches under final tree: 64
R128 64
R64 32
R32 16
R16 8
QF 4
SF 2
F 1

YEAR: 2023
Final: Carlos Alcaraz def. Novak Djokovic
R128 leaf matches under final tree: 64
R128 64
R64 32
R32 16
R16 8
QF 4
SF 2
F 1

YEAR: 2024
Final: Carlos Alcaraz def. Novak Djokovic
R128 leaf matches under final tree: 64
R128 64
R64 32
R32 16
R16 8
QF 4
SF 2
F 1

YEAR: 2025
Final: Jannik Sinner def. Carlos Alcaraz
R128 leaf matches under final tree: 64
R128 64
R64 32
R32 16
R16 8
QF 4
SF 2
F 1


In [176]:
def get_wimbledon_player_features(match_df, year):
    """
    Creates a dictionary of player features using each player's R128 pre-match values.
    
    Key idea:
    For bracket prediction, we want each player's information before Wimbledon started.
    The R128 rows contain each player's pre-match Elo/form/rank values.
    """
    
    wim = match_df[
        (match_df["tourney_name"].str.contains("Wimbledon", case=False, na=False)) &
        (match_df["tourney_date"].dt.year == year)
    ].copy()
    
    r128 = wim[wim["round"] == "R128"].copy()
    
    players = {}
    
    for _, row in r128.iterrows():
        
        # Winner from the actual R128 match
        players[row["winner_id"]] = {
            "player_id": row["winner_id"],
            "name": row["winner_name"],
            "rank": row["winner_rank"],
            "rank_points": row["winner_rank_points"],
            
            "elo": row["w_elo_pre"],
            "surface_elo": row["w_surface_elo_pre"],
            "score_elo": row["w_score_elo_pre"],
            "score_surface_elo": row["w_score_surface_elo_pre"],
            
            "last10_win_pct": row["w_last10_win_pct_pre"],
            "last20_win_pct": row["w_last20_win_pct_pre"],
            "surface_last10_win_pct": row["w_surface_last10_win_pct_pre"],
            "surface_last20_win_pct": row["w_surface_last20_win_pct_pre"],
            
            "last10_matches": row["w_last10_matches_pre"],
            "last20_matches": row["w_last20_matches_pre"],
            "surface_last10_matches": row["w_surface_last10_matches_pre"],
            "surface_last20_matches": row["w_surface_last20_matches_pre"],
        }
        
        # Loser from the actual R128 match
        players[row["loser_id"]] = {
            "player_id": row["loser_id"],
            "name": row["loser_name"],
            "rank": row["loser_rank"],
            "rank_points": row["loser_rank_points"],
            
            "elo": row["l_elo_pre"],
            "surface_elo": row["l_surface_elo_pre"],
            "score_elo": row["l_score_elo_pre"],
            "score_surface_elo": row["l_score_surface_elo_pre"],
            
            "last10_win_pct": row["l_last10_win_pct_pre"],
            "last20_win_pct": row["l_last20_win_pct_pre"],
            "surface_last10_win_pct": row["l_surface_last10_win_pct_pre"],
            "surface_last20_win_pct": row["l_surface_last20_win_pct_pre"],
            
            "last10_matches": row["l_last10_matches_pre"],
            "last20_matches": row["l_last20_matches_pre"],
            "surface_last10_matches": row["l_surface_last10_matches_pre"],
            "surface_last20_matches": row["l_surface_last20_matches_pre"],
        }
    
    return players

In [177]:
both_elo_feature_cols = [
    "elo_diff",
    "surface_elo_diff",
    "score_elo_diff",
    "score_surface_elo_diff",
    "rank_diff",
    "log_rank_points_diff",
    "last10_win_pct_pre_diff",
    "last20_win_pct_pre_diff",
    "surface_last10_win_pct_pre_diff",
    "surface_last20_win_pct_pre_diff",
    "last10_matches_pre_diff",
    "last20_matches_pre_diff",
    "surface_last10_matches_pre_diff",
    "surface_last20_matches_pre_diff"
]
def make_matchup_features(player_A, player_B, feature_cols):
    """
    Converts two player dictionaries into one model-ready row.
    
    The output is from Player A's perspective:
    positive values usually mean Player A has the advantage.
    """
    
    row = {
        "elo_diff": player_A["elo"] - player_B["elo"],
        "surface_elo_diff": player_A["surface_elo"] - player_B["surface_elo"],
        "score_elo_diff": player_A["score_elo"] - player_B["score_elo"],
        "score_surface_elo_diff": player_A["score_surface_elo"] - player_B["score_surface_elo"],
        
        # Better rank is smaller, so use B - A
        "rank_diff": player_B["rank"] - player_A["rank"],
        
        "log_rank_points_diff": (
            np.log1p(player_A["rank_points"]) - np.log1p(player_B["rank_points"])
        ),
        
        "last10_win_pct_pre_diff": (
            player_A["last10_win_pct"] - player_B["last10_win_pct"]
        ),
        
        "last20_win_pct_pre_diff": (
            player_A["last20_win_pct"] - player_B["last20_win_pct"]
        ),
        
        "surface_last10_win_pct_pre_diff": (
            player_A["surface_last10_win_pct"] - player_B["surface_last10_win_pct"]
        ),
        
        "surface_last20_win_pct_pre_diff": (
            player_A["surface_last20_win_pct"] - player_B["surface_last20_win_pct"]
        ),
        
        "last10_matches_pre_diff": (
            player_A["last10_matches"] - player_B["last10_matches"]
        ),
        
        "last20_matches_pre_diff": (
            player_A["last20_matches"] - player_B["last20_matches"]
        ),
        
        "surface_last10_matches_pre_diff": (
            player_A["surface_last10_matches"] - player_B["surface_last10_matches"]
        ),
        
        "surface_last20_matches_pre_diff": (
            player_A["surface_last20_matches"] - player_B["surface_last20_matches"]
        ),
    }
    
    return pd.DataFrame([row])[feature_cols]

In [178]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def train_model_before_wimbledon(neutral_df, year, feature_cols):
    """
    Trains the model only on matches before that year's Wimbledon started.
    """
    
    wim = neutral_df[
        (neutral_df["tourney_name"].str.contains("Wimbledon", case=False, na=False)) &
        (neutral_df["date"].dt.year == year)
    ].copy()
    
    cutoff_date = wim["date"].min()
    
    train = neutral_df[
        neutral_df["date"] < cutoff_date
    ].dropna(subset=feature_cols + ["A_won"]).copy()
    
    X_train = train[feature_cols]
    y_train = train["A_won"]
    
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(max_iter=5000))
    ])
    
    model.fit(X_train, y_train)
    
    print("Training cutoff:", cutoff_date)
    print("Training matches:", len(train))
    
    return model

model_2024 = train_model_before_wimbledon(
    neutral_df,
    2024,
    both_elo_feature_cols
)

Training cutoff: 2024-07-01 00:00:00
Training matches: 17287


In [179]:
def predict_player_A_win_prob(model, player_A, player_B, feature_cols):
    """
    Predicts probability that Player A beats Player B.
    """
    
    X = make_matchup_features(player_A, player_B, feature_cols)
    
    return model.predict_proba(X)[:, 1][0]

In [180]:
def generate_predicted_bracket_from_tree(node, players, model, feature_cols, picks=None):
    """
    Recursively predicts winners through a reconstructed Wimbledon bracket tree.
    
    For each match/node:
    - If it is R128, use the actual first-round players.
    - If it is later than R128, first predict the winners from the two feeder branches.
    - Then predict the winner between those two predicted players.
    """
    
    if picks is None:
        picks = []
    
    # Base case: actual first-round matchup
    if node["round"] == "R128":
        p1 = players[node["winner_id"]]
        p2 = players[node["loser_id"]]
    
    # Recursive case: later-round matchup
    else:
        p1, picks = generate_predicted_bracket_from_tree(
            node["left"],
            players,
            model,
            feature_cols,
            picks
        )
        
        p2, picks = generate_predicted_bracket_from_tree(
            node["right"],
            players,
            model,
            feature_cols,
            picks
        )
    
    # Predict probability that p1 beats p2
    p1_win_prob = predict_player_A_win_prob(
        model,
        p1,
        p2,
        feature_cols
    )
    
    if p1_win_prob >= 0.5:
        pick = p1
        pick_prob = p1_win_prob
    else:
        pick = p2
        pick_prob = 1 - p1_win_prob
    
    picks.append({
        "round": node["round"],
        "round_index": node["round_index"],
        "player_1": p1["name"],
        "player_2": p2["name"],
        "pick_id": pick["player_id"],
        "pick_name": pick["name"],
        "pick_prob": pick_prob,
        "actual_winner_id": node["winner_id"],
        "actual_winner_name": node["winner_name"]
    })
    
    return pick, picks

In [181]:
root_2024, nodes_2024 = build_wimbledon_bracket_tree(match_df, 2024)

players_2024 = get_wimbledon_player_features(match_df, 2024)

model_2024 = train_model_before_wimbledon(
    neutral_df,
    2024,
    both_elo_feature_cols
)

champion_pick_2024, picks_2024 = generate_predicted_bracket_from_tree(
    root_2024,
    players_2024,
    model_2024,
    both_elo_feature_cols
)

print("Predicted champion:", champion_pick_2024["name"])

Training cutoff: 2024-07-01 00:00:00
Training matches: 17287
Predicted champion: Novak Djokovic


In [182]:
picks_2024_df = pd.DataFrame(picks_2024)

picks_2024_df.head()

picks_2024_df["round"].value_counts()

round_order = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7
}

picks_2024_df["round_order"] = picks_2024_df["round"].map(round_order)

picks_2024_df.sort_values(["round_order", "round_index"]).tail(15)

,round,round_index,player_1,player_2,pick_id,pick_name,pick_prob,actual_winner_id,actual_winner_name,round_order
45,R16,0,Carlos Alcaraz,Ugo Humbert,A0E2,Carlos Alcaraz,0.760564,A0E2,Carlos Alcaraz,4
14,R16,1,Jannik Sinner,Ben Shelton,S0AG,Jannik Sinner,0.833664,S0AG,Jannik Sinner,4
29,R16,2,Grigor Dimitrov,Daniil Medvedev,MM58,Daniil Medvedev,0.542453,MM58,Daniil Medvedev,4
60,R16,3,Tommy Paul,Casper Ruud,PL56,Tommy Paul,0.535497,PL56,Tommy Paul,4
108,R16,4,Stefanos Tsitsipas,Andrey Rublev,RE44,Andrey Rublev,0.516638,M0EJ,Lorenzo Musetti,4
77,R16,5,Alex de Minaur,Hubert Hurkacz,DH58,Alex de Minaur,0.536143,DH58,Alex de Minaur,4
123,R16,6,Alexander Zverev,Taylor Fritz,Z355,Alexander Zverev,0.572779,FB98,Taylor Fritz,4
92,R16,7,Karen Khachanov,Novak Djokovic,D643,Novak Djokovic,0.833905,D643,Novak Djokovic,4
30,QF,0,Jannik Sinner,Daniil Medvedev,S0AG,Jannik Sinner,0.649056,MM58,Daniil Medvedev,5
61,QF,1,Carlos Alcaraz,Tommy Paul,A0E2,Carlos Alcaraz,0.642784,A0E2,Carlos Alcaraz,5


In [183]:
round_points = {
    "R128": 10,
    "R64": 20,
    "R32": 40,
    "R16": 80,
    "QF": 160,
    "SF": 320,
    "F": 640
}

picks_2024_df["correct"] = (
    picks_2024_df["pick_id"] == picks_2024_df["actual_winner_id"]
)

picks_2024_df["points_possible"] = picks_2024_df["round"].map(round_points)

picks_2024_df["points_earned"] = np.where(
    picks_2024_df["correct"],
    picks_2024_df["points_possible"],
    0
)

score_2024_by_round = picks_2024_df.groupby("round").agg(
    correct=("correct", "sum"),
    total=("correct", "size"),
    accuracy=("correct", "mean"),
    points=("points_earned", "sum"),
    possible=("points_possible", "sum")
).reset_index()

score_2024_by_round["score_pct"] = (
    score_2024_by_round["points"] / score_2024_by_round["possible"]
)

round_order = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7
}

score_2024_by_round["round_order"] = score_2024_by_round["round"].map(round_order)

score_2024_by_round = score_2024_by_round.sort_values("round_order")

score_2024_by_round

,round,correct,total,accuracy,points,possible,score_pct,round_order
2,R128,49,64,0.765625,490,640,0.765625,1
5,R64,19,32,0.593750,380,640,0.593750,2
4,R32,11,16,0.687500,440,640,0.687500,3
3,R16,6,8,0.750000,480,640,0.750000,4
1,QF,2,4,0.500000,320,640,0.500000,5
6,SF,1,2,0.500000,320,640,0.500000,6
0,F,0,1,0.000000,0,640,0.000000,7


In [184]:
total_score_2024 = picks_2024_df["points_earned"].sum()
possible_score_2024 = picks_2024_df["points_possible"].sum()

print("2024 score:", total_score_2024)
print("Possible:", possible_score_2024)
print("Score %:", total_score_2024 / possible_score_2024)
print("Pick accuracy:", picks_2024_df["correct"].mean())
print("Predicted champion:", picks_2024_df.loc[picks_2024_df["round"] == "F", "pick_name"].iloc[0])
print("Actual champion:", picks_2024_df.loc[picks_2024_df["round"] == "F", "actual_winner_name"].iloc[0])

2024 score: 2430
Possible: 4480
Score %: 0.5424107142857143
Pick accuracy: 0.6929133858267716
Predicted champion: Novak Djokovic
Actual champion: Carlos Alcaraz


In [185]:
picks_2024_df[
    picks_2024_df["correct"] == False
].sort_values("points_possible", ascending=False)[[
    "round",
    "player_1",
    "player_2",
    "pick_name",
    "pick_prob",
    "actual_winner_name",
    "points_possible"
]].head(20)

,round,player_1,player_2,pick_name,pick_prob,actual_winner_name,points_possible
126,F,Jannik Sinner,Novak Djokovic,Novak Djokovic,0.562991,Carlos Alcaraz,640
62,SF,Jannik Sinner,Carlos Alcaraz,Jannik Sinner,0.559066,Carlos Alcaraz,320
30,QF,Jannik Sinner,Daniil Medvedev,Jannik Sinner,0.649056,Daniil Medvedev,160
124,QF,Andrey Rublev,Alexander Zverev,Alexander Zverev,0.566057,Lorenzo Musetti,160
123,R16,Alexander Zverev,Taylor Fritz,Alexander Zverev,0.572779,Taylor Fritz,80
108,R16,Stefanos Tsitsipas,Andrey Rublev,Andrey Rublev,0.516638,Lorenzo Musetti,80
59,R32,Casper Ruud,Roberto Bautista Agut,Casper Ruud,0.690653,Roberto Bautista Agut,40
100,R32,Stefanos Tsitsipas,Sebastian Korda,Stefanos Tsitsipas,0.557484,Giovanni Mpetshi Perricard,40
107,R32,Andrey Rublev,Lorenzo Musetti,Andrey Rublev,0.663342,Lorenzo Musetti,40
76,R32,Hubert Hurkacz,Francisco Cerundolo,Hubert Hurkacz,0.753081,Arthur Fils,40


In [186]:
def generate_and_score_tree_bracket(match_df, neutral_df, year, feature_cols, round_points):
    """
    Builds the bracket tree for a Wimbledon year, trains the model before that Wimbledon,
    generates predicted picks, and scores the bracket.
    """
    
    # 1. Build actual bracket tree structure
    root, nodes_by_round = build_wimbledon_bracket_tree(match_df, year)
    
    # 2. Get pre-Wimbledon player features
    players = get_wimbledon_player_features(match_df, year)
    
    # 3. Train model only on matches before that Wimbledon
    model = train_model_before_wimbledon(
        neutral_df,
        year,
        feature_cols
    )
    
    # 4. Generate predicted bracket
    champion_pick, picks = generate_predicted_bracket_from_tree(
        root,
        players,
        model,
        feature_cols
    )
    
    picks_df = pd.DataFrame(picks)
    picks_df["year"] = year
    
    # 5. Score picks
    picks_df["correct"] = (
        picks_df["pick_id"] == picks_df["actual_winner_id"]
    )
    
    picks_df["points_possible"] = picks_df["round"].map(round_points)
    
    picks_df["points_earned"] = np.where(
        picks_df["correct"],
        picks_df["points_possible"],
        0
    )
    
    return picks_df

In [187]:
def backtest_tree_bracket_scores(match_df, neutral_df, years, feature_cols, round_points):
    all_year_summaries = []
    all_scored_brackets = []
    
    for year in years:
        print(f"Running {year}...")
        
        scored = generate_and_score_tree_bracket(
            match_df,
            neutral_df,
            year,
            feature_cols,
            round_points
        )
        
        champion_row = scored[scored["round"] == "F"].iloc[0]
        
        summary = {
            "year": year,
            "score": scored["points_earned"].sum(),
            "possible": scored["points_possible"].sum(),
            "score_pct": scored["points_earned"].sum() / scored["points_possible"].sum(),
            "correct_picks": scored["correct"].sum(),
            "total_picks": len(scored),
            "pick_accuracy": scored["correct"].mean(),
            "predicted_champion": champion_row["pick_name"],
            "actual_champion": champion_row["actual_winner_name"],
            "champion_correct": champion_row["correct"]
        }
        
        all_year_summaries.append(summary)
        all_scored_brackets.append(scored)
    
    results_df = pd.DataFrame(all_year_summaries)
    scored_brackets_df = pd.concat(all_scored_brackets, ignore_index=True)
    
    return results_df, scored_brackets_df

In [188]:
wim_years = [2019, 2021, 2022, 2023, 2024, 2025]

round_points = {
    "R128": 10,
    "R64": 20,
    "R32": 40,
    "R16": 80,
    "QF": 160,
    "SF": 320,
    "F": 640
}

tree_bracket_results, tree_scored_brackets = backtest_tree_bracket_scores(
    match_df,
    neutral_df,
    wim_years,
    both_elo_feature_cols,
    round_points
)

tree_bracket_results

Running 2019...
Training cutoff: 2019-07-01 00:00:00
Training matches: 4414
Running 2021...
Training cutoff: 2021-06-28 00:00:00
Training matches: 8506
Running 2022...
Training cutoff: 2022-06-27 00:00:00
Training matches: 11363
Running 2023...
Training cutoff: 2023-07-03 00:00:00
Training matches: 14317
Running 2024...
Training cutoff: 2024-07-01 00:00:00
Training matches: 17287
Running 2025...
Training cutoff: 2025-06-30 00:00:00
Training matches: 20224


,year,score,possible,score_pct,correct_picks,total_picks,pick_accuracy,predicted_champion,actual_champion,champion_correct
0,2019,2840,4480,0.633929,80,127,0.629921,Novak Djokovic,Novak Djokovic,True
1,2021,2810,4480,0.627232,84,127,0.661417,Novak Djokovic,Novak Djokovic,True
2,2022,2590,4480,0.578125,81,127,0.637795,Novak Djokovic,Novak Djokovic,True
3,2023,2570,4480,0.573661,79,127,0.622047,Novak Djokovic,Carlos Alcaraz,False
4,2024,2430,4480,0.542411,88,127,0.692913,Novak Djokovic,Carlos Alcaraz,False
5,2025,2370,4480,0.529018,68,127,0.535433,Carlos Alcaraz,Jannik Sinner,False


In [189]:
print(tree_bracket_results.mean(numeric_only=True))
round_score_summary = tree_scored_brackets.groupby("round").agg(
    correct=("correct", "sum"),
    total=("correct", "size"),
    accuracy=("correct", "mean"),
    points=("points_earned", "sum"),
    possible=("points_possible", "sum")
).reset_index()

round_score_summary["score_pct"] = (
    round_score_summary["points"] / round_score_summary["possible"]
)

round_order = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7
}

round_score_summary["round_order"] = round_score_summary["round"].map(round_order)
round_score_summary = round_score_summary.sort_values("round_order")

round_score_summary

year                2022.333333
score               2601.666667
possible            4480.000000
score_pct              0.580729
correct_picks         80.000000
total_picks          127.000000
pick_accuracy          0.629921
champion_correct       0.500000
dtype: float64


,round,correct,total,accuracy,points,possible,score_pct,round_order
2,R128,273,384,0.710938,2730,3840,0.710938,1
5,R64,108,192,0.562500,2160,3840,0.562500,2
4,R32,50,96,0.520833,2000,3840,0.520833,3
3,R16,23,48,0.479167,1840,3840,0.479167,4
1,QF,15,24,0.625000,2400,3840,0.625000,5
6,SF,8,12,0.666667,2560,3840,0.666667,6
0,F,3,6,0.500000,1920,3840,0.500000,7


In [190]:
both_elo_h2h_feature_cols = [
    "elo_diff",
    "surface_elo_diff",
    "score_elo_diff",
    "score_surface_elo_diff",
    "rank_diff",
    "log_rank_points_diff",
    "last10_win_pct_pre_diff",
    "last20_win_pct_pre_diff",
    "surface_last10_win_pct_pre_diff",
    "surface_last20_win_pct_pre_diff",
    "last10_matches_pre_diff",
    "last20_matches_pre_diff",
    "surface_last10_matches_pre_diff",
    "surface_last20_matches_pre_diff",
    "h2h_win_pct_diff",
    "h2h_matches_pre",
    "h2h_weighted_advantage_diff"
]

In [191]:
h2h_wim_results, h2h_wim_preds = backtest_wimbledon_model(
    neutral_df,
    both_elo_h2h_feature_cols,
    years=wim_years,
    model_name="both_elo_h2h_model"
)

h2h_wim_results

,model,year,wimbledon_start,train_matches,test_matches,accuracy,log_loss,brier
0,both_elo_h2h_model,2019,2019-07-01,4414,127,0.740157,0.549020,0.186307
1,both_elo_h2h_model,2021,2021-06-28,8506,127,0.740157,0.547049,0.183437
2,both_elo_h2h_model,2022,2022-06-27,11363,127,0.755906,0.545984,0.182742
3,both_elo_h2h_model,2023,2023-07-03,14317,127,0.724409,0.553088,0.187026
4,both_elo_h2h_model,2024,2024-07-01,17287,127,0.779528,0.513643,0.169183
5,both_elo_h2h_model,2025,2025-06-30,20224,127,0.677165,0.583074,0.199932


In [192]:
def build_h2h_lookup_before_date(match_df, cutoff_date):
    """
    Builds H2H records using only matches before cutoff_date.
    
    Returns:
        h2h_wins[(player_id, opponent_id)] = number of wins by player_id over opponent_id
    """
    prior_matches = match_df[match_df["tourney_date"] < cutoff_date].copy()
    
    h2h_wins = {}
    
    for _, row in prior_matches.iterrows():
        winner = row["winner_id"]
        loser = row["loser_id"]
        
        h2h_wins[(winner, loser)] = h2h_wins.get((winner, loser), 0) + 1
    
    return h2h_wins


def get_h2h_features(player_A, player_B, h2h_wins):
    """
    Creates H2H features from Player A's perspective.
    """
    A_id = player_A["player_id"]
    B_id = player_B["player_id"]
    
    A_wins = h2h_wins.get((A_id, B_id), 0)
    B_wins = h2h_wins.get((B_id, A_id), 0)
    
    total = A_wins + B_wins
    
    if total > 0:
        A_win_pct = A_wins / total
        B_win_pct = B_wins / total
    else:
        A_win_pct = 0.5
        B_win_pct = 0.5
    
    h2h_win_pct_diff = A_win_pct - B_win_pct
    
    A_adv = A_win_pct - 0.5
    B_adv = B_win_pct - 0.5
    
    A_weighted_adv = A_adv * np.log1p(total)
    B_weighted_adv = B_adv * np.log1p(total)
    
    h2h_weighted_advantage_diff = A_weighted_adv - B_weighted_adv
    
    return {
        "h2h_win_pct_diff": h2h_win_pct_diff,
        "h2h_matches_pre": total,
        "h2h_weighted_advantage_diff": h2h_weighted_advantage_diff
    }


def make_matchup_features_with_h2h(player_A, player_B, feature_cols, h2h_wins):
    """
    Converts two players into a model-ready feature row, including H2H features.
    """
    
    row = {
        "elo_diff": player_A["elo"] - player_B["elo"],
        "surface_elo_diff": player_A["surface_elo"] - player_B["surface_elo"],
        "score_elo_diff": player_A["score_elo"] - player_B["score_elo"],
        "score_surface_elo_diff": player_A["score_surface_elo"] - player_B["score_surface_elo"],
        
        "rank_diff": player_B["rank"] - player_A["rank"],
        
        "log_rank_points_diff": (
            np.log1p(player_A["rank_points"]) - np.log1p(player_B["rank_points"])
        ),
        
        "last10_win_pct_pre_diff": player_A["last10_win_pct"] - player_B["last10_win_pct"],
        "last20_win_pct_pre_diff": player_A["last20_win_pct"] - player_B["last20_win_pct"],
        "surface_last10_win_pct_pre_diff": player_A["surface_last10_win_pct"] - player_B["surface_last10_win_pct"],
        "surface_last20_win_pct_pre_diff": player_A["surface_last20_win_pct"] - player_B["surface_last20_win_pct"],
        
        "last10_matches_pre_diff": player_A["last10_matches"] - player_B["last10_matches"],
        "last20_matches_pre_diff": player_A["last20_matches"] - player_B["last20_matches"],
        "surface_last10_matches_pre_diff": player_A["surface_last10_matches"] - player_B["surface_last10_matches"],
        "surface_last20_matches_pre_diff": player_A["surface_last20_matches"] - player_B["surface_last20_matches"],
    }
    
    row.update(get_h2h_features(player_A, player_B, h2h_wins))
    
    return pd.DataFrame([row])[feature_cols]

In [193]:
def predict_player_A_win_prob_with_h2h(model, player_A, player_B, feature_cols, h2h_wins):
    """
    Predicts probability that Player A beats Player B, including H2H features.
    """
    
    X = make_matchup_features_with_h2h(
        player_A,
        player_B,
        feature_cols,
        h2h_wins
    )
    
    return model.predict_proba(X)[:, 1][0]

players_2024 = get_wimbledon_player_features(match_df, 2024)

wim_2024 = neutral_df[
    (neutral_df["tourney_name"].str.contains("Wimbledon", case=False, na=False)) &
    (neutral_df["date"].dt.year == 2024)
].copy()

cutoff_2024 = wim_2024["date"].min()

h2h_wins_2024 = build_h2h_lookup_before_date(match_df, cutoff_2024)

model_2024_h2h = train_model_before_wimbledon(
    neutral_df,
    2024,
    both_elo_h2h_feature_cols
)

sample_players = list(players_2024.values())



Training cutoff: 2024-07-01 00:00:00
Training matches: 17287


In [194]:
def generate_predicted_bracket_from_tree_with_h2h(
    node, 
    players, 
    model, 
    feature_cols, 
    h2h_wins,
    picks=None
):
    """
    Recursively predicts winners through the bracket tree using H2H features.
    
    For each match:
    - get the two players
    - calculate matchup features including H2H
    - predict the winner
    - advance the predicted winner
    """
    
    if picks is None:
        picks = []
    
    # Base case: actual first-round matchup
    if node["round"] == "R128":
        p1 = players[node["winner_id"]]
        p2 = players[node["loser_id"]]
    
    # Recursive case: predict winners of feeder branches first
    else:
        p1, picks = generate_predicted_bracket_from_tree_with_h2h(
            node["left"],
            players,
            model,
            feature_cols,
            h2h_wins,
            picks
        )
        
        p2, picks = generate_predicted_bracket_from_tree_with_h2h(
            node["right"],
            players,
            model,
            feature_cols,
            h2h_wins,
            picks
        )
    
    # Predict probability that p1 beats p2, including H2H
    p1_win_prob = predict_player_A_win_prob_with_h2h(
        model,
        p1,
        p2,
        feature_cols,
        h2h_wins
    )
    
    if p1_win_prob >= 0.5:
        pick = p1
        pick_prob = p1_win_prob
    else:
        pick = p2
        pick_prob = 1 - p1_win_prob
    
    picks.append({
        "round": node["round"],
        "round_index": node["round_index"],
        "player_1": p1["name"],
        "player_2": p2["name"],
        "pick_id": pick["player_id"],
        "pick_name": pick["name"],
        "pick_prob": pick_prob,
        "actual_winner_id": node["winner_id"],
        "actual_winner_name": node["winner_name"],
        "h2h_matches_pre": get_h2h_features(p1, p2, h2h_wins)["h2h_matches_pre"],
        "h2h_weighted_advantage_diff": get_h2h_features(p1, p2, h2h_wins)["h2h_weighted_advantage_diff"]
    })
    
    return pick, picks

In [195]:
root_2024, nodes_2024 = build_wimbledon_bracket_tree(match_df, 2024)

players_2024 = get_wimbledon_player_features(match_df, 2024)

wim_2024 = neutral_df[
    (neutral_df["tourney_name"].str.contains("Wimbledon", case=False, na=False)) &
    (neutral_df["date"].dt.year == 2024)
].copy()

cutoff_2024 = wim_2024["date"].min()

h2h_wins_2024 = build_h2h_lookup_before_date(match_df, cutoff_2024)

model_2024_h2h = train_model_before_wimbledon(
    neutral_df,
    2024,
    both_elo_h2h_feature_cols
)

champion_pick_2024_h2h, picks_2024_h2h = generate_predicted_bracket_from_tree_with_h2h(
    root_2024,
    players_2024,
    model_2024_h2h,
    both_elo_h2h_feature_cols,
    h2h_wins_2024
)

print("Predicted champion:", champion_pick_2024_h2h["name"])

Training cutoff: 2024-07-01 00:00:00
Training matches: 17287
Predicted champion: Novak Djokovic


In [196]:
picks_2024_h2h_df = pd.DataFrame(picks_2024_h2h)

picks_2024_h2h_df.head()

,round,round_index,player_1,player_2,pick_id,pick_name,pick_prob,actual_winner_id,actual_winner_name,h2h_matches_pre,h2h_weighted_advantage_diff
0,R128,26,Tallon Griekspoor,Daniel Elahi Galan,GJ37,Tallon Griekspoor,0.704505,GJ37,Tallon Griekspoor,0,0.000000
1,R128,30,Miomir Kecmanovic,Sumit Nagal,KI95,Miomir Kecmanovic,0.687161,KI95,Miomir Kecmanovic,1,0.693147
2,R64,9,Tallon Griekspoor,Miomir Kecmanovic,GJ37,Tallon Griekspoor,0.573331,KI95,Miomir Kecmanovic,1,0.693147
3,R128,10,Matteo Berrettini,Marton Fucsovics,BK40,Matteo Berrettini,0.618657,BK40,Matteo Berrettini,1,-0.693147
4,R128,27,Jannik Sinner,Yannick Hanfmann,S0AG,Jannik Sinner,0.916592,S0AG,Jannik Sinner,1,0.693147


In [197]:
round_points = {
    "R128": 10,
    "R64": 20,
    "R32": 40,
    "R16": 80,
    "QF": 160,
    "SF": 320,
    "F": 640
}

picks_2024_h2h_df["correct"] = (
    picks_2024_h2h_df["pick_id"] == picks_2024_h2h_df["actual_winner_id"]
)

picks_2024_h2h_df["points_possible"] = (
    picks_2024_h2h_df["round"].map(round_points)
)

picks_2024_h2h_df["points_earned"] = np.where(
    picks_2024_h2h_df["correct"],
    picks_2024_h2h_df["points_possible"],
    0
)

In [198]:
round_order = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7
}

score_2024_h2h_by_round = picks_2024_h2h_df.groupby("round").agg(
    correct=("correct", "sum"),
    total=("correct", "size"),
    accuracy=("correct", "mean"),
    points=("points_earned", "sum"),
    possible=("points_possible", "sum")
).reset_index()

score_2024_h2h_by_round["score_pct"] = (
    score_2024_h2h_by_round["points"] / score_2024_h2h_by_round["possible"]
)

score_2024_h2h_by_round["round_order"] = (
    score_2024_h2h_by_round["round"].map(round_order)
)

score_2024_h2h_by_round = score_2024_h2h_by_round.sort_values("round_order")

score_2024_h2h_by_round

,round,correct,total,accuracy,points,possible,score_pct,round_order
2,R128,50,64,0.78125,500,640,0.78125,1
5,R64,19,32,0.59375,380,640,0.59375,2
4,R32,11,16,0.68750,440,640,0.68750,3
3,R16,6,8,0.75000,480,640,0.75000,4
1,QF,2,4,0.50000,320,640,0.50000,5
6,SF,1,2,0.50000,320,640,0.50000,6
0,F,0,1,0.00000,0,640,0.00000,7


In [199]:
total_score_2024_h2h = picks_2024_h2h_df["points_earned"].sum()
possible_score_2024_h2h = picks_2024_h2h_df["points_possible"].sum()

print("2024 H2H score:", total_score_2024_h2h)
print("Possible:", possible_score_2024_h2h)
print("Score %:", total_score_2024_h2h / possible_score_2024_h2h)
print("Pick accuracy:", picks_2024_h2h_df["correct"].mean())
print("Predicted champion:", picks_2024_h2h_df.loc[picks_2024_h2h_df["round"] == "F", "pick_name"].iloc[0])
print("Actual champion:", picks_2024_h2h_df.loc[picks_2024_h2h_df["round"] == "F", "actual_winner_name"].iloc[0])

2024 H2H score: 2440
Possible: 4480
Score %: 0.5446428571428571
Pick accuracy: 0.7007874015748031
Predicted champion: Novak Djokovic
Actual champion: Carlos Alcaraz


In [200]:
comparison_2024 = pd.DataFrame([
    {
        "model": "both_elo_no_h2h",
        "score": picks_2024_df["points_earned"].sum(),
        "score_pct": picks_2024_df["points_earned"].sum() / picks_2024_df["points_possible"].sum(),
        "pick_accuracy": picks_2024_df["correct"].mean(),
        "predicted_champion": picks_2024_df.loc[picks_2024_df["round"] == "F", "pick_name"].iloc[0],
        "actual_champion": picks_2024_df.loc[picks_2024_df["round"] == "F", "actual_winner_name"].iloc[0],
        "champion_correct": picks_2024_df.loc[picks_2024_df["round"] == "F", "correct"].iloc[0]
    },
    {
        "model": "both_elo_h2h",
        "score": picks_2024_h2h_df["points_earned"].sum(),
        "score_pct": picks_2024_h2h_df["points_earned"].sum() / picks_2024_h2h_df["points_possible"].sum(),
        "pick_accuracy": picks_2024_h2h_df["correct"].mean(),
        "predicted_champion": picks_2024_h2h_df.loc[picks_2024_h2h_df["round"] == "F", "pick_name"].iloc[0],
        "actual_champion": picks_2024_h2h_df.loc[picks_2024_h2h_df["round"] == "F", "actual_winner_name"].iloc[0],
        "champion_correct": picks_2024_h2h_df.loc[picks_2024_h2h_df["round"] == "F", "correct"].iloc[0]
    }
])

comparison_2024

,model,score,score_pct,pick_accuracy,predicted_champion,actual_champion,champion_correct
0,both_elo_no_h2h,2430,0.542411,0.692913,Novak Djokovic,Carlos Alcaraz,False
1,both_elo_h2h,2440,0.544643,0.700787,Novak Djokovic,Carlos Alcaraz,False


In [201]:
def generate_and_score_tree_bracket_with_h2h(match_df, neutral_df, year, feature_cols, round_points):
    """
    Builds the bracket tree, trains the H2H model before Wimbledon,
    generates H2H-aware predicted picks, and scores the bracket.
    """
    
    root, nodes_by_round = build_wimbledon_bracket_tree(match_df, year)
    players = get_wimbledon_player_features(match_df, year)
    
    wim = neutral_df[
        (neutral_df["tourney_name"].str.contains("Wimbledon", case=False, na=False)) &
        (neutral_df["date"].dt.year == year)
    ].copy()
    
    cutoff_date = wim["date"].min()
    h2h_wins = build_h2h_lookup_before_date(match_df, cutoff_date)
    
    model = train_model_before_wimbledon(
        neutral_df,
        year,
        feature_cols
    )
    
    champion_pick, picks = generate_predicted_bracket_from_tree_with_h2h(
        root,
        players,
        model,
        feature_cols,
        h2h_wins
    )
    
    picks_df = pd.DataFrame(picks)
    picks_df["year"] = year
    
    picks_df["correct"] = (
        picks_df["pick_id"] == picks_df["actual_winner_id"]
    )
    
    picks_df["points_possible"] = picks_df["round"].map(round_points)
    
    picks_df["points_earned"] = np.where(
        picks_df["correct"],
        picks_df["points_possible"],
        0
    )
    
    return picks_df

def backtest_tree_bracket_scores_with_h2h(match_df, neutral_df, years, feature_cols, round_points):
    all_year_summaries = []
    all_scored_brackets = []
    
    for year in years:
        print(f"Running {year}...")
        
        scored = generate_and_score_tree_bracket_with_h2h(
            match_df,
            neutral_df,
            year,
            feature_cols,
            round_points
        )
        
        champion_row = scored[scored["round"] == "F"].iloc[0]
        
        summary = {
            "year": year,
            "score": scored["points_earned"].sum(),
            "possible": scored["points_possible"].sum(),
            "score_pct": scored["points_earned"].sum() / scored["points_possible"].sum(),
            "correct_picks": scored["correct"].sum(),
            "total_picks": len(scored),
            "pick_accuracy": scored["correct"].mean(),
            "predicted_champion": champion_row["pick_name"],
            "actual_champion": champion_row["actual_winner_name"],
            "champion_correct": champion_row["correct"]
        }
        
        all_year_summaries.append(summary)
        all_scored_brackets.append(scored)
    
    results_df = pd.DataFrame(all_year_summaries)
    scored_brackets_df = pd.concat(all_scored_brackets, ignore_index=True)
    
    return results_df, scored_brackets_df

In [202]:
h2h_bracket_results, h2h_scored_brackets = backtest_tree_bracket_scores_with_h2h(
    match_df,
    neutral_df,
    wim_years,
    both_elo_h2h_feature_cols,
    round_points
)

h2h_bracket_results

Running 2019...
Training cutoff: 2019-07-01 00:00:00
Training matches: 4414
Running 2021...
Training cutoff: 2021-06-28 00:00:00
Training matches: 8506
Running 2022...
Training cutoff: 2022-06-27 00:00:00
Training matches: 11363
Running 2023...
Training cutoff: 2023-07-03 00:00:00
Training matches: 14317
Running 2024...
Training cutoff: 2024-07-01 00:00:00
Training matches: 17287
Running 2025...
Training cutoff: 2025-06-30 00:00:00
Training matches: 20224


,year,score,possible,score_pct,correct_picks,total_picks,pick_accuracy,predicted_champion,actual_champion,champion_correct
0,2019,2840,4480,0.633929,80,127,0.629921,Novak Djokovic,Novak Djokovic,True
1,2021,2810,4480,0.627232,84,127,0.661417,Novak Djokovic,Novak Djokovic,True
2,2022,2590,4480,0.578125,81,127,0.637795,Novak Djokovic,Novak Djokovic,True
3,2023,2550,4480,0.569196,78,127,0.614173,Novak Djokovic,Carlos Alcaraz,False
4,2024,2440,4480,0.544643,89,127,0.700787,Novak Djokovic,Carlos Alcaraz,False
5,2025,2380,4480,0.531250,69,127,0.543307,Carlos Alcaraz,Jannik Sinner,False


In [203]:
bracket_model_comparison = pd.concat([
    tree_bracket_results.assign(model="both_elo_no_h2h"),
    h2h_bracket_results.assign(model="both_elo_h2h")
])

bracket_model_comparison.groupby("model").agg(
    avg_score=("score", "mean"),
    avg_score_pct=("score_pct", "mean"),
    avg_pick_accuracy=("pick_accuracy", "mean"),
    champion_accuracy=("champion_correct", "mean")
).reset_index()

,model,avg_score,avg_score_pct,avg_pick_accuracy,champion_accuracy
0,both_elo_h2h,2601.666667,0.580729,0.631234,0.5
1,both_elo_no_h2h,2601.666667,0.580729,0.629921,0.5


Where did H2H change picks?

In [204]:
h2h_vs_no_h2h_picks = tree_scored_brackets.merge(
    h2h_scored_brackets,
    on=["year", "round", "round_index"],
    suffixes=("_no_h2h", "_h2h")
)

changed_picks = h2h_vs_no_h2h_picks[
    h2h_vs_no_h2h_picks["pick_id_no_h2h"] != h2h_vs_no_h2h_picks["pick_id_h2h"]
].copy()

changed_picks[[
    "year",
    "round",
    "player_1_no_h2h",
    "player_2_no_h2h",
    "pick_name_no_h2h",
    "correct_no_h2h",
    "pick_name_h2h",
    "correct_h2h",
    "actual_winner_name_no_h2h",
    "points_possible_no_h2h"
]].sort_values(["year", "points_possible_no_h2h"], ascending=[True, False])

,year,round,player_1_no_h2h,player_2_no_h2h,pick_name_no_h2h,correct_no_h2h,pick_name_h2h,correct_h2h,actual_winner_name_no_h2h,points_possible_no_h2h
7,2019,R128,Tennys Sandgren,Yasutaka Uchiyama,Tennys Sandgren,True,Yasutaka Uchiyama,False,Tennys Sandgren,10
70,2019,R128,Steve Darcis,Mischa Zverev,Mischa Zverev,False,Steve Darcis,True,Steve Darcis,10
132,2021,R64,Alejandro Davidovich Fokina,Andreas Seppi,Andreas Seppi,False,Alejandro Davidovich Fokina,False,Denis Kudla,20
436,2023,R64,Ben Shelton,Laslo Djere,Laslo Djere,True,Maxime Cressy,False,Laslo Djere,20
435,2023,R128,Laslo Djere,Maxime Cressy,Laslo Djere,True,Maxime Cressy,False,Laslo Djere,10
452,2023,R128,Oscar Otte,Dominik Koepfer,Dominik Koepfer,False,Oscar Otte,True,Oscar Otte,10
547,2024,R128,Brandon Nakashima,Sebastian Baez,Sebastian Baez,False,Brandon Nakashima,True,Brandon Nakashima,10
722,2025,R64,Frances Tiafoe,Roberto Bautista Agut,Frances Tiafoe,False,Roberto Bautista Agut,False,Cameron Norrie,20
674,2025,R128,Arthur Cazaux,Adam Walton,Adam Walton,False,Arthur Cazaux,True,Arthur Cazaux,10


H2H Does not make a large difference, as these are all early round matches. Now we will optimize expected bracket score rather than picking the favorite in each projected match

### Optimizing Expected Bracket Score

Cached Matchup Probability Helper

In [205]:
def get_match_win_prob_cached(model, player_A, player_B, feature_cols, prob_cache):
    """
    Returns P(Player A beats Player B), using a cache so we don't recompute
    the same matchup probability many times.
    """
    
    A_id = player_A["player_id"]
    B_id = player_B["player_id"]
    
    key = (A_id, B_id)
    
    if key not in prob_cache:
        p_A_wins = predict_player_A_win_prob(
            model,
            player_A,
            player_B,
            feature_cols
        )
        
        prob_cache[(A_id, B_id)] = p_A_wins
        prob_cache[(B_id, A_id)] = 1 - p_A_wins
    
    return prob_cache[key]

Expected score optimizer

In [206]:
def solve_expected_score_bracket(node, players, model, feature_cols, round_points, prob_cache=None):
    """
    Solves for the bracket picks that maximize expected bracket score.
    
    Returns:
        node_win_probs:
            Dict mapping player_id -> probability that player wins this node.
            
        best_by_winner:
            Dict mapping picked winner_id -> best bracket subtree if that player
            is picked to win this node.
    """
    
    if prob_cache is None:
        prob_cache = {}
    
    current_round = node["round"]
    current_points = round_points[current_round]
    
    # Base case: first-round match
    if current_round == "R128":
        p1 = players[node["winner_id"]]
        p2 = players[node["loser_id"]]
        
        p1_win_prob = get_match_win_prob_cached(
            model,
            p1,
            p2,
            feature_cols,
            prob_cache
        )
        
        node_win_probs = {
            p1["player_id"]: p1_win_prob,
            p2["player_id"]: 1 - p1_win_prob
        }
        
        best_by_winner = {}
        
        for p in [p1, p2]:
            pid = p["player_id"]
            win_prob = node_win_probs[pid]
            expected_points = current_points * win_prob
            
            pick = {
                "round": current_round,
                "round_index": node["round_index"],
                "player_1": p1["name"],
                "player_2": p2["name"],
                "pick_id": pid,
                "pick_name": p["name"],
                "win_prob": win_prob,
                "expected_points": expected_points,
                "actual_winner_id": node["winner_id"],
                "actual_winner_name": node["winner_name"]
            }
            
            best_by_winner[pid] = {
                "player": p,
                "expected_score": expected_points,
                "picks": [pick]
            }
        
        return node_win_probs, best_by_winner
    
    
    # Recursive case: solve left and right sides first
    left_probs, left_best = solve_expected_score_bracket(
        node["left"],
        players,
        model,
        feature_cols,
        round_points,
        prob_cache
    )
    
    right_probs, right_best = solve_expected_score_bracket(
        node["right"],
        players,
        model,
        feature_cols,
        round_points,
        prob_cache
    )
    
    # Compute probability each possible player wins this node
    node_win_probs = {}
    
    # Players coming from left side
    for left_id, p_left_reaches in left_probs.items():
        left_player = players[left_id]
        
        prob_wins_node_given_reaches = 0
        
        for right_id, p_right_reaches in right_probs.items():
            right_player = players[right_id]
            
            p_left_beats_right = get_match_win_prob_cached(
                model,
                left_player,
                right_player,
                feature_cols,
                prob_cache
            )
            
            prob_wins_node_given_reaches += (
                p_right_reaches * p_left_beats_right
            )
        
        node_win_probs[left_id] = (
            p_left_reaches * prob_wins_node_given_reaches
        )
    
    # Players coming from right side
    for right_id, p_right_reaches in right_probs.items():
        right_player = players[right_id]
        
        prob_wins_node_given_reaches = 0
        
        for left_id, p_left_reaches in left_probs.items():
            left_player = players[left_id]
            
            p_right_beats_left = get_match_win_prob_cached(
                model,
                right_player,
                left_player,
                feature_cols,
                prob_cache
            )
            
            prob_wins_node_given_reaches += (
                p_left_reaches * p_right_beats_left
            )
        
        node_win_probs[right_id] = (
            p_right_reaches * prob_wins_node_given_reaches
        )
    
    # Best possible bracket from each child, regardless of who wins that child
    best_left_overall = max(
        left_best.values(),
        key=lambda x: x["expected_score"]
    )
    
    best_right_overall = max(
        right_best.values(),
        key=lambda x: x["expected_score"]
    )
    
    best_by_winner = {}
    
    # If we pick someone from the left side to win this node
    for left_id, left_option in left_best.items():
        picked_player = left_option["player"]
        opponent_pick = best_right_overall["player"]
        
        win_prob = node_win_probs[left_id]
        expected_points = current_points * win_prob
        
        current_pick = {
            "round": current_round,
            "round_index": node["round_index"],
            "player_1": picked_player["name"],
            "player_2": opponent_pick["name"],
            "pick_id": left_id,
            "pick_name": picked_player["name"],
            "win_prob": win_prob,
            "expected_points": expected_points,
            "actual_winner_id": node["winner_id"],
            "actual_winner_name": node["winner_name"]
        }
        
        total_expected_score = (
            left_option["expected_score"]
            + best_right_overall["expected_score"]
            + expected_points
        )
        
        best_by_winner[left_id] = {
            "player": picked_player,
            "expected_score": total_expected_score,
            "picks": (
                left_option["picks"]
                + best_right_overall["picks"]
                + [current_pick]
            )
        }
    
    # If we pick someone from the right side to win this node
    for right_id, right_option in right_best.items():
        picked_player = right_option["player"]
        opponent_pick = best_left_overall["player"]
        
        win_prob = node_win_probs[right_id]
        expected_points = current_points * win_prob
        
        current_pick = {
            "round": current_round,
            "round_index": node["round_index"],
            "player_1": opponent_pick["name"],
            "player_2": picked_player["name"],
            "pick_id": right_id,
            "pick_name": picked_player["name"],
            "win_prob": win_prob,
            "expected_points": expected_points,
            "actual_winner_id": node["winner_id"],
            "actual_winner_name": node["winner_name"]
        }
        
        total_expected_score = (
            best_left_overall["expected_score"]
            + right_option["expected_score"]
            + expected_points
        )
        
        best_by_winner[right_id] = {
            "player": picked_player,
            "expected_score": total_expected_score,
            "picks": (
                best_left_overall["picks"]
                + right_option["picks"]
                + [current_pick]
            )
        }
    
    return node_win_probs, best_by_winner

In [207]:
root_2024, nodes_2024 = build_wimbledon_bracket_tree(match_df, 2024)

players_2024 = get_wimbledon_player_features(match_df, 2024)

model_2024 = train_model_before_wimbledon(
    neutral_df,
    2024,
    both_elo_feature_cols
)

root_win_probs_2024, best_by_champion_2024 = solve_expected_score_bracket(
    root_2024,
    players_2024,
    model_2024,
    both_elo_feature_cols,
    round_points
)

Training cutoff: 2024-07-01 00:00:00
Training matches: 17287


In [209]:
champ_probs_2024 = pd.DataFrame([
    {
        "player_id": pid,
        "player_name": players_2024[pid]["name"],
        "championship_prob": prob
    }
    for pid, prob in root_win_probs_2024.items()
]).sort_values("championship_prob", ascending=False)

champ_probs_2024.head(15)

,player_id,player_name,championship_prob
94,D643,Novak Djokovic,0.291407
6,S0AG,Jannik Sinner,0.160032
36,A0E2,Carlos Alcaraz,0.129858
24,MM58,Daniil Medvedev,0.052578
116,Z355,Alexander Zverev,0.045790
64,DH58,Alex de Minaur,0.043523
72,HB71,Hubert Hurkacz,0.035389
54,PL56,Tommy Paul,0.033036
105,RE44,Andrey Rublev,0.028463
18,D875,Grigor Dimitrov,0.028430


In [215]:
best_champion_option_2024 = max(
    best_by_champion_2024.values(),
    key=lambda x: x["expected_score"]
)

optimized_picks_2024_df = pd.DataFrame(
    best_champion_option_2024["picks"]
)

print("Optimized champion pick:", best_champion_option_2024["player"]["name"])
print("Optimized expected score:", best_champion_option_2024["expected_score"])

optimized_picks_2024_df["round"].value_counts()

Optimized champion pick: Novak Djokovic
Optimized expected score: 2067.3067125474445


round
R128    64
R64     32
R32     16
R16      8
QF       4
SF       2
F        1
Name: count, dtype: int64

In [216]:
optimized_picks_2024_df["correct"] = (
    optimized_picks_2024_df["pick_id"] == optimized_picks_2024_df["actual_winner_id"]
)

optimized_picks_2024_df["points_possible"] = (
    optimized_picks_2024_df["round"].map(round_points)
)

optimized_picks_2024_df["points_earned"] = np.where(
    optimized_picks_2024_df["correct"],
    optimized_picks_2024_df["points_possible"],
    0
)

In [217]:
optimized_2024_by_round = optimized_picks_2024_df.groupby("round").agg(
    correct=("correct", "sum"),
    total=("correct", "size"),
    accuracy=("correct", "mean"),
    points=("points_earned", "sum"),
    possible=("points_possible", "sum")
).reset_index()

optimized_2024_by_round["score_pct"] = (
    optimized_2024_by_round["points"] / optimized_2024_by_round["possible"]
)

optimized_2024_by_round["round_order"] = (
    optimized_2024_by_round["round"].map(round_order)
)

optimized_2024_by_round = optimized_2024_by_round.sort_values("round_order")

optimized_2024_by_round

,round,correct,total,accuracy,points,possible,score_pct,round_order
2,R128,49,64,0.765625,490,640,0.765625,1
5,R64,19,32,0.593750,380,640,0.593750,2
4,R32,11,16,0.687500,440,640,0.687500,3
3,R16,6,8,0.750000,480,640,0.750000,4
1,QF,2,4,0.500000,320,640,0.500000,5
6,SF,1,2,0.500000,320,640,0.500000,6
0,F,0,1,0.000000,0,640,0.000000,7


In [218]:
optimized_score_2024 = optimized_picks_2024_df["points_earned"].sum()
optimized_possible_2024 = optimized_picks_2024_df["points_possible"].sum()

print("Optimized 2024 score:", optimized_score_2024)
print("Possible:", optimized_possible_2024)
print("Score %:", optimized_score_2024 / optimized_possible_2024)
print("Pick accuracy:", optimized_picks_2024_df["correct"].mean())
print("Predicted champion:", optimized_picks_2024_df.loc[optimized_picks_2024_df["round"] == "F", "pick_name"].iloc[0])
print("Actual champion:", optimized_picks_2024_df.loc[optimized_picks_2024_df["round"] == "F", "actual_winner_name"].iloc[0])

Optimized 2024 score: 2430
Possible: 4480
Score %: 0.5424107142857143
Pick accuracy: 0.6929133858267716
Predicted champion: Novak Djokovic
Actual champion: Carlos Alcaraz


In [219]:
comparison_2024_optimized = pd.DataFrame([
    {
        "model": "favorite_pick_bracket",
        "score": picks_2024_df["points_earned"].sum(),
        "score_pct": picks_2024_df["points_earned"].sum() / picks_2024_df["points_possible"].sum(),
        "pick_accuracy": picks_2024_df["correct"].mean(),
        "predicted_champion": picks_2024_df.loc[picks_2024_df["round"] == "F", "pick_name"].iloc[0],
        "actual_champion": picks_2024_df.loc[picks_2024_df["round"] == "F", "actual_winner_name"].iloc[0],
        "champion_correct": picks_2024_df.loc[picks_2024_df["round"] == "F", "correct"].iloc[0]
    },
    {
        "model": "expected_score_optimized",
        "score": optimized_picks_2024_df["points_earned"].sum(),
        "score_pct": optimized_picks_2024_df["points_earned"].sum() / optimized_picks_2024_df["points_possible"].sum(),
        "pick_accuracy": optimized_picks_2024_df["correct"].mean(),
        "predicted_champion": optimized_picks_2024_df.loc[optimized_picks_2024_df["round"] == "F", "pick_name"].iloc[0],
        "actual_champion": optimized_picks_2024_df.loc[optimized_picks_2024_df["round"] == "F", "actual_winner_name"].iloc[0],
        "champion_correct": optimized_picks_2024_df.loc[optimized_picks_2024_df["round"] == "F", "correct"].iloc[0]
    }
])

comparison_2024_optimized

,model,score,score_pct,pick_accuracy,predicted_champion,actual_champion,champion_correct
0,favorite_pick_bracket,2430,0.542411,0.692913,Novak Djokovic,Carlos Alcaraz,False
1,expected_score_optimized,2430,0.542411,0.692913,Novak Djokovic,Carlos Alcaraz,False


In [220]:
def generate_and_score_optimized_tree_bracket(match_df, neutral_df, year, feature_cols, round_points):
    """
    Builds the Wimbledon bracket tree, trains the model before Wimbledon,
    solves the expected-score optimized bracket, and scores it.
    """
    
    root, nodes_by_round = build_wimbledon_bracket_tree(match_df, year)
    players = get_wimbledon_player_features(match_df, year)
    
    model = train_model_before_wimbledon(
        neutral_df,
        year,
        feature_cols
    )
    
    root_win_probs, best_by_champion = solve_expected_score_bracket(
        root,
        players,
        model,
        feature_cols,
        round_points
    )
    
    best_champion_option = max(
        best_by_champion.values(),
        key=lambda x: x["expected_score"]
    )
    
    picks_df = pd.DataFrame(best_champion_option["picks"])
    picks_df["year"] = year
    picks_df["optimized_expected_score"] = best_champion_option["expected_score"]
    
    picks_df["correct"] = (
        picks_df["pick_id"] == picks_df["actual_winner_id"]
    )
    
    picks_df["points_possible"] = picks_df["round"].map(round_points)
    
    picks_df["points_earned"] = np.where(
        picks_df["correct"],
        picks_df["points_possible"],
        0
    )
    
    return picks_df, root_win_probs, best_champion_option

In [221]:
def backtest_optimized_tree_bracket_scores(match_df, neutral_df, years, feature_cols, round_points):
    all_year_summaries = []
    all_scored_brackets = []
    all_champ_probs = []
    
    for year in years:
        print(f"Running {year}...")
        
        scored, root_win_probs, best_champion_option = generate_and_score_optimized_tree_bracket(
            match_df,
            neutral_df,
            year,
            feature_cols,
            round_points
        )
        
        champion_row = scored[scored["round"] == "F"].iloc[0]
        
        summary = {
            "year": year,
            "score": scored["points_earned"].sum(),
            "possible": scored["points_possible"].sum(),
            "score_pct": scored["points_earned"].sum() / scored["points_possible"].sum(),
            "correct_picks": scored["correct"].sum(),
            "total_picks": len(scored),
            "pick_accuracy": scored["correct"].mean(),
            "predicted_champion": champion_row["pick_name"],
            "actual_champion": champion_row["actual_winner_name"],
            "champion_correct": champion_row["correct"],
            "optimized_expected_score": best_champion_option["expected_score"]
        }
        
        all_year_summaries.append(summary)
        all_scored_brackets.append(scored)
        
        players = get_wimbledon_player_features(match_df, year)
        
        champ_probs_year = pd.DataFrame([
            {
                "year": year,
                "player_id": pid,
                "player_name": players[pid]["name"],
                "championship_prob": prob
            }
            for pid, prob in root_win_probs.items()
        ])
        
        all_champ_probs.append(champ_probs_year)
    
    results_df = pd.DataFrame(all_year_summaries)
    scored_brackets_df = pd.concat(all_scored_brackets, ignore_index=True)
    champ_probs_df = pd.concat(all_champ_probs, ignore_index=True)
    
    return results_df, scored_brackets_df, champ_probs_df

In [222]:
optimized_bracket_results, optimized_scored_brackets, optimized_champ_probs = backtest_optimized_tree_bracket_scores(
    match_df,
    neutral_df,
    wim_years,
    both_elo_feature_cols,
    round_points
)

optimized_bracket_results

Running 2019...
Training cutoff: 2019-07-01 00:00:00
Training matches: 4414
Running 2021...
Training cutoff: 2021-06-28 00:00:00
Training matches: 8506
Running 2022...
Training cutoff: 2022-06-27 00:00:00
Training matches: 11363
Running 2023...
Training cutoff: 2023-07-03 00:00:00
Training matches: 14317
Running 2024...
Training cutoff: 2024-07-01 00:00:00
Training matches: 17287
Running 2025...
Training cutoff: 2025-06-30 00:00:00
Training matches: 20224


,year,score,possible,score_pct,correct_picks,total_picks,pick_accuracy,predicted_champion,actual_champion,champion_correct,optimized_expected_score
0,2019,3180,4480,0.709821,81,127,0.637795,Novak Djokovic,Novak Djokovic,True,2175.011830
1,2021,2850,4480,0.636161,85,127,0.669291,Novak Djokovic,Novak Djokovic,True,2074.798573
2,2022,2630,4480,0.587054,82,127,0.645669,Novak Djokovic,Novak Djokovic,True,2106.188048
3,2023,2850,4480,0.636161,80,127,0.629921,Novak Djokovic,Carlos Alcaraz,False,2047.817748
4,2024,2430,4480,0.542411,88,127,0.692913,Novak Djokovic,Carlos Alcaraz,False,2067.306713
5,2025,2370,4480,0.529018,68,127,0.535433,Carlos Alcaraz,Jannik Sinner,False,2087.479811


In [223]:
bracket_optimization_comparison = pd.concat([
    tree_bracket_results.assign(model="favorite_pick"),
    optimized_bracket_results.assign(model="expected_score_optimized")
])

bracket_optimization_comparison.groupby("model").agg(
    avg_score=("score", "mean"),
    avg_score_pct=("score_pct", "mean"),
    avg_pick_accuracy=("pick_accuracy", "mean"),
    champion_accuracy=("champion_correct", "mean"),
    avg_expected_score=("optimized_expected_score", "mean")
).reset_index()

,model,avg_score,avg_score_pct,avg_pick_accuracy,champion_accuracy,avg_expected_score
0,expected_score_optimized,2718.333333,0.606771,0.635171,0.5,2093.100454
1,favorite_pick,2601.666667,0.580729,0.629921,0.5,NaN


In [224]:
optimized_vs_favorite_picks = tree_scored_brackets.merge(
    optimized_scored_brackets,
    on=["year", "round", "round_index"],
    suffixes=("_favorite", "_optimized")
)

optimized_changed_picks = optimized_vs_favorite_picks[
    optimized_vs_favorite_picks["pick_id_favorite"] != optimized_vs_favorite_picks["pick_id_optimized"]
].copy()

optimized_changed_picks[[
    "year",
    "round",
    "player_1_favorite",
    "player_2_favorite",
    "pick_name_favorite",
    "correct_favorite",
    "pick_name_optimized",
    "correct_optimized",
    "actual_winner_name_favorite",
    "points_possible_favorite"
]].sort_values(["year", "points_possible_favorite"], ascending=[True, False])

,year,round,player_1_favorite,player_2_favorite,pick_name_favorite,correct_favorite,pick_name_optimized,correct_optimized,actual_winner_name_favorite,points_possible_favorite
62,2019,SF,Rafael Nadal,Roger Federer,Rafael Nadal,False,Roger Federer,True,Roger Federer,320
92,2019,R16,Milos Raonic,Kevin Anderson,Kevin Anderson,False,Milos Raonic,False,Guido Pella,80
76,2019,R32,Roberto Bautista Agut,Karen Khachanov,Karen Khachanov,False,Roberto Bautista Agut,True,Roberto Bautista Agut,40
115,2019,R32,Gael Monfils,Felix Auger-Aliassime,Gael Monfils,False,Felix Auger-Aliassime,False,Ugo Humbert,40
33,2019,R64,Taylor Fritz,Jan-Lennard Struff,Jan-Lennard Struff,True,Taylor Fritz,False,Jan-Lennard Struff,20
218,2021,R32,Lorenzo Sonego,Pablo Carreno Busta,Pablo Carreno Busta,False,Lorenzo Sonego,True,Lorenzo Sonego,40
298,2022,R32,Frances Tiafoe,Pablo Carreno Busta,Pablo Carreno Busta,False,Frances Tiafoe,True,Frances Tiafoe,40
474,2023,QF,Taylor Fritz,Casper Ruud,Taylor Fritz,False,Jannik Sinner,True,Jannik Sinner,160
410,2023,R16,Frances Tiafoe,Holger Rune,Frances Tiafoe,False,Holger Rune,True,Holger Rune,80
458,2023,R16,Jannik Sinner,Taylor Fritz,Taylor Fritz,False,Jannik Sinner,True,Jannik Sinner,80


## Testing optimized model against naive model that picks off of who has more ranking points

In [230]:
def generate_rank_points_bracket_from_tree(node, players, picks=None):
    """
    Recursively generates a bracket by always picking the player
    with more ranking points.
    """
    
    if picks is None:
        picks = []
    
    if node["round"] == "R128":
        p1 = players[node["winner_id"]]
        p2 = players[node["loser_id"]]
    else:
        p1, picks = generate_rank_points_bracket_from_tree(
            node["left"],
            players,
            picks
        )
        
        p2, picks = generate_rank_points_bracket_from_tree(
            node["right"],
            players,
            picks
        )
    
    p1_points = p1["rank_points"]
    p2_points = p2["rank_points"]
    
    # If one player has missing rank points, treat them as 0
    if pd.isna(p1_points):
        p1_points = 0
    if pd.isna(p2_points):
        p2_points = 0
    
    if p1_points >= p2_points:
        pick = p1
    else:
        pick = p2
    
    picks.append({
        "round": node["round"],
        "round_index": node["round_index"],
        "player_1": p1["name"],
        "player_2": p2["name"],
        "pick_id": pick["player_id"],
        "pick_name": pick["name"],
        "actual_winner_id": node["winner_id"],
        "actual_winner_name": node["winner_name"]
    })
    
    return pick, picks

def generate_and_score_rank_points_bracket(match_df, year, round_points):
    root, nodes_by_round = build_wimbledon_bracket_tree(match_df, year)
    players = get_wimbledon_player_features(match_df, year)
    
    champion_pick, picks = generate_rank_points_bracket_from_tree(
        root,
        players
    )
    
    picks_df = pd.DataFrame(picks)
    picks_df["year"] = year
    
    picks_df["correct"] = (
        picks_df["pick_id"] == picks_df["actual_winner_id"]
    )
    
    picks_df["points_possible"] = picks_df["round"].map(round_points)
    
    picks_df["points_earned"] = np.where(
        picks_df["correct"],
        picks_df["points_possible"],
        0
    )
    
    return picks_df

def backtest_rank_points_bracket_scores(match_df, years, round_points):
    all_year_summaries = []
    all_scored_brackets = []
    
    for year in years:
        print(f"Running {year}...")
        
        scored = generate_and_score_rank_points_bracket(
            match_df,
            year,
            round_points
        )
        
        champion_row = scored[scored["round"] == "F"].iloc[0]
        
        summary = {
            "year": year,
            "score": scored["points_earned"].sum(),
            "possible": scored["points_possible"].sum(),
            "score_pct": scored["points_earned"].sum() / scored["points_possible"].sum(),
            "correct_picks": scored["correct"].sum(),
            "total_picks": len(scored),
            "pick_accuracy": scored["correct"].mean(),
            "predicted_champion": champion_row["pick_name"],
            "actual_champion": champion_row["actual_winner_name"],
            "champion_correct": champion_row["correct"]
        }
        
        all_year_summaries.append(summary)
        all_scored_brackets.append(scored)
    
    results_df = pd.DataFrame(all_year_summaries)
    scored_brackets_df = pd.concat(all_scored_brackets, ignore_index=True)
    
    return results_df, scored_brackets_df

In [231]:
rank_points_results, rank_points_scored_brackets = backtest_rank_points_bracket_scores(
    match_df,
    wim_years,
    round_points
)

rank_points_results

Running 2019...
Running 2021...
Running 2022...
Running 2023...
Running 2024...
Running 2025...


,year,score,possible,score_pct,correct_picks,total_picks,pick_accuracy,predicted_champion,actual_champion,champion_correct
0,2019,2800,4480,0.625000,78,127,0.614173,Novak Djokovic,Novak Djokovic,True
1,2021,2590,4480,0.578125,80,127,0.629921,Novak Djokovic,Novak Djokovic,True
2,2022,2520,4480,0.562500,75,127,0.590551,Novak Djokovic,Novak Djokovic,True
3,2023,3240,4480,0.723214,74,127,0.582677,Carlos Alcaraz,Carlos Alcaraz,True
4,2024,2210,4480,0.493304,80,127,0.629921,Jannik Sinner,Carlos Alcaraz,False
5,2025,2820,4480,0.629464,65,127,0.511811,Jannik Sinner,Jannik Sinner,True


In [232]:
favorite_results_for_compare = tree_bracket_results.copy()
favorite_results_for_compare["model"] = "model_favorite_pick"

optimized_results_for_compare = optimized_bracket_results.copy()
optimized_results_for_compare["model"] = "model_expected_score_optimized"

rank_points_results_for_compare = rank_points_results.copy()
rank_points_results_for_compare["model"] = "rank_points_baseline"

all_bracket_models = pd.concat([
    rank_points_results_for_compare,
    favorite_results_for_compare,
    optimized_results_for_compare
], ignore_index=True)

all_bracket_models.groupby("model").agg(
    avg_score=("score", "mean"),
    avg_score_pct=("score_pct", "mean"),
    avg_pick_accuracy=("pick_accuracy", "mean"),
    champion_accuracy=("champion_correct", "mean")
).reset_index()

,model,avg_score,avg_score_pct,avg_pick_accuracy,champion_accuracy
0,model_expected_score_optimized,2718.333333,0.606771,0.635171,0.500000
1,model_favorite_pick,2601.666667,0.580729,0.629921,0.500000
2,rank_points_baseline,2696.666667,0.601935,0.593176,0.833333


Interestingly, the highest ranked player has not won Wimbledon only once in the past 7 years

In [233]:
optimized_vs_rank = optimized_scored_brackets.merge(
    rank_points_scored_brackets,
    on=["year", "round", "round_index"],
    suffixes=("_optimized", "_rank")
)

optimized_vs_rank_changed = optimized_vs_rank[
    optimized_vs_rank["pick_id_optimized"] != optimized_vs_rank["pick_id_rank"]
].copy()

optimized_vs_rank_changed[[
    "year",
    "round",
    "player_1_optimized",
    "player_2_optimized",
    "pick_name_optimized",
    "correct_optimized",
    "pick_name_rank",
    "correct_rank",
    "actual_winner_name_optimized",
    "points_possible_optimized"
]].sort_values(
    ["year", "points_possible_optimized"],
    ascending=[True, False]
)

,year,round,player_1_optimized,player_2_optimized,pick_name_optimized,correct_optimized,pick_name_rank,correct_rank,actual_winner_name_optimized,points_possible_optimized
62,2019,SF,Rafael Nadal,Roger Federer,Roger Federer,True,Rafael Nadal,False,Roger Federer,320
92,2019,R16,Milos Raonic,Kevin Anderson,Milos Raonic,False,Kevin Anderson,False,Guido Pella,80
21,2019,R32,Nikoloz Basilashvili,Marin Cilic,Marin Cilic,False,Nikoloz Basilashvili,False,Joao Sousa,40
76,2019,R32,Roberto Bautista Agut,Karen Khachanov,Roberto Bautista Agut,True,Karen Khachanov,False,Roberto Bautista Agut,40
115,2019,R32,Gael Monfils,Felix Auger-Aliassime,Felix Auger-Aliassime,False,Gael Monfils,False,Ugo Humbert,40
...,...,...,...,...,...,...,...,...,...,...
670,2025,R128,Jesper de Jong,Christopher Eubanks,Christopher Eubanks,False,Jesper de Jong,True,Jesper de Jong,10
702,2025,R128,Jan-Lennard Struff,Filip Misolic,Jan-Lennard Struff,True,Filip Misolic,False,Jan-Lennard Struff,10
709,2025,R128,Adrian Mannarino,Christopher O'Connell,Adrian Mannarino,True,Christopher O'Connell,False,Adrian Mannarino,10
737,2025,R128,Luciano Darderi,Roman Safiullin,Roman Safiullin,False,Luciano Darderi,True,Luciano Darderi,10


In [234]:
optimized_vs_rank_changed_summary = optimized_vs_rank_changed.groupby("round").agg(
    changed_picks=("round", "size"),
    optimized_correct=("correct_optimized", "sum"),
    rank_correct=("correct_rank", "sum"),
    optimized_points=("points_earned_optimized", "sum"),
    rank_points=("points_earned_rank", "sum")
).reset_index()

optimized_vs_rank_changed_summary["point_difference"] = (
    optimized_vs_rank_changed_summary["optimized_points"]
    - optimized_vs_rank_changed_summary["rank_points"]
)

optimized_vs_rank_changed_summary["round_order"] = (
    optimized_vs_rank_changed_summary["round"].map(round_order)
)

optimized_vs_rank_changed_summary = optimized_vs_rank_changed_summary.sort_values("round_order")

optimized_vs_rank_changed_summary

,round,changed_picks,optimized_correct,rank_correct,optimized_points,rank_points,point_difference,round_order
2,R128,43,31,12,310,120,190,1
5,R64,12,7,2,140,40,100,2
4,R32,15,6,2,240,80,160,3
3,R16,4,2,0,160,0,160,4
1,QF,4,3,0,480,0,480,5
6,SF,1,1,0,320,0,320,6
0,F,3,0,2,0,1280,-1280,7


Although highest ranked player won many times, based on current context, we cannot assume Sinner will win automatically.

## Backtest of French Open 2026 just for fun, because there were so many early upsets

In [225]:
def build_tournament_bracket_tree(match_df, tournament_name, year):
    tournament = match_df[
        (match_df["tourney_name"].str.contains(tournament_name, case=False, na=False)) &
        (match_df["tourney_date"].dt.year == year)
    ].copy()
    
    print("Tournament matches:", len(tournament))
    print("Surface:", tournament["surface"].unique())
    print("Rounds:", tournament["round"].value_counts().to_dict())
    
    nodes_by_round = {}
    
    for rnd in ROUND_SEQUENCE:
        round_matches = tournament[tournament["round"] == rnd].copy().reset_index(drop=True)
        nodes = []
        
        for idx, row in round_matches.iterrows():
            node = {
                "round": rnd,
                "round_index": idx,
                "winner_id": row["winner_id"],
                "winner_name": row["winner_name"],
                "loser_id": row["loser_id"],
                "loser_name": row["loser_name"],
                "player_ids": {row["winner_id"], row["loser_id"]},
                "left": None,
                "right": None,
                "row": row
            }
            nodes.append(node)
        
        nodes_by_round[rnd] = nodes
    
    for i in range(1, len(ROUND_SEQUENCE)):
        prev_round = ROUND_SEQUENCE[i - 1]
        curr_round = ROUND_SEQUENCE[i]
        
        prev_nodes = nodes_by_round[prev_round]
        curr_nodes = nodes_by_round[curr_round]
        
        for curr_node in curr_nodes:
            feeders = []
            
            for prev_node in prev_nodes:
                if prev_node["winner_id"] in curr_node["player_ids"]:
                    feeders.append(prev_node)
            
            if len(feeders) != 2:
                print(
                    f"WARNING: {year} {tournament_name} {curr_round} match "
                    f"{curr_node['round_index']} has {len(feeders)} feeders instead of 2"
                )
            
            if len(feeders) >= 1:
                curr_node["left"] = feeders[0]
            if len(feeders) >= 2:
                curr_node["right"] = feeders[1]
    
    final_nodes = nodes_by_round["F"]
    
    if len(final_nodes) != 1:
        raise ValueError(f"Expected exactly one final node, found {len(final_nodes)}")
    
    return final_nodes[0], nodes_by_round

def get_tournament_player_features(match_df, tournament_name, year):
    tournament = match_df[
        (match_df["tourney_name"].str.contains(tournament_name, case=False, na=False)) &
        (match_df["tourney_date"].dt.year == year)
    ].copy()
    
    r128 = tournament[tournament["round"] == "R128"].copy()
    
    players = {}
    
    for _, row in r128.iterrows():
        players[row["winner_id"]] = {
            "player_id": row["winner_id"],
            "name": row["winner_name"],
            "rank": row["winner_rank"],
            "rank_points": row["winner_rank_points"],
            "elo": row["w_elo_pre"],
            "surface_elo": row["w_surface_elo_pre"],
            "score_elo": row["w_score_elo_pre"],
            "score_surface_elo": row["w_score_surface_elo_pre"],
            "last10_win_pct": row["w_last10_win_pct_pre"],
            "last20_win_pct": row["w_last20_win_pct_pre"],
            "surface_last10_win_pct": row["w_surface_last10_win_pct_pre"],
            "surface_last20_win_pct": row["w_surface_last20_win_pct_pre"],
            "last10_matches": row["w_last10_matches_pre"],
            "last20_matches": row["w_last20_matches_pre"],
            "surface_last10_matches": row["w_surface_last10_matches_pre"],
            "surface_last20_matches": row["w_surface_last20_matches_pre"],
        }
        
        players[row["loser_id"]] = {
            "player_id": row["loser_id"],
            "name": row["loser_name"],
            "rank": row["loser_rank"],
            "rank_points": row["loser_rank_points"],
            "elo": row["l_elo_pre"],
            "surface_elo": row["l_surface_elo_pre"],
            "score_elo": row["l_score_elo_pre"],
            "score_surface_elo": row["l_score_surface_elo_pre"],
            "last10_win_pct": row["l_last10_win_pct_pre"],
            "last20_win_pct": row["l_last20_win_pct_pre"],
            "surface_last10_win_pct": row["l_surface_last10_win_pct_pre"],
            "surface_last20_win_pct": row["l_surface_last20_win_pct_pre"],
            "last10_matches": row["l_last10_matches_pre"],
            "last20_matches": row["l_last20_matches_pre"],
            "surface_last10_matches": row["l_surface_last10_matches_pre"],
            "surface_last20_matches": row["l_surface_last20_matches_pre"],
        }
    
    return players

def train_model_before_tournament(neutral_df, tournament_name, year, feature_cols):
    tournament = neutral_df[
        (neutral_df["tourney_name"].str.contains(tournament_name, case=False, na=False)) &
        (neutral_df["date"].dt.year == year)
    ].copy()
    
    cutoff_date = tournament["date"].min()
    
    train = neutral_df[
        neutral_df["date"] < cutoff_date
    ].dropna(subset=feature_cols + ["A_won"]).copy()
    
    X_train = train[feature_cols]
    y_train = train["A_won"]
    
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(max_iter=5000))
    ])
    
    model.fit(X_train, y_train)
    
    print("Tournament:", tournament_name, year)
    print("Training cutoff:", cutoff_date)
    print("Training matches:", len(train))
    
    return model

def generate_and_score_optimized_tournament_bracket(
    match_df,
    neutral_df,
    tournament_name,
    year,
    feature_cols,
    round_points
):
    root, nodes_by_round = build_tournament_bracket_tree(
        match_df,
        tournament_name,
        year
    )
    
    players = get_tournament_player_features(
        match_df,
        tournament_name,
        year
    )
    
    model = train_model_before_tournament(
        neutral_df,
        tournament_name,
        year,
        feature_cols
    )
    
    root_win_probs, best_by_champion = solve_expected_score_bracket(
        root,
        players,
        model,
        feature_cols,
        round_points
    )
    
    best_champion_option = max(
        best_by_champion.values(),
        key=lambda x: x["expected_score"]
    )
    
    picks_df = pd.DataFrame(best_champion_option["picks"])
    picks_df["year"] = year
    picks_df["tournament"] = tournament_name
    picks_df["optimized_expected_score"] = best_champion_option["expected_score"]
    
    picks_df["correct"] = (
        picks_df["pick_id"] == picks_df["actual_winner_id"]
    )
    
    picks_df["points_possible"] = picks_df["round"].map(round_points)
    
    picks_df["points_earned"] = np.where(
        picks_df["correct"],
        picks_df["points_possible"],
        0
    )
    
    return picks_df, root_win_probs, best_champion_option

In [226]:
rg_2026_picks, rg_2026_champ_probs, rg_2026_best_option = generate_and_score_optimized_tournament_bracket(
    match_df,
    neutral_df,
    "Roland Garros",
    2026,
    both_elo_feature_cols,
    round_points
)

Tournament matches: 127
Surface: ['Clay']
Rounds: {'R128': 64, 'R64': 32, 'R32': 16, 'R16': 8, 'QF': 4, 'SF': 2, 'F': 1}
Tournament: Roland Garros 2026
Training cutoff: 2026-05-24 00:00:00
Training matches: 22769


In [227]:
rg_2026_score = rg_2026_picks["points_earned"].sum()
rg_2026_possible = rg_2026_picks["points_possible"].sum()

print("Roland Garros 2026 optimized score:", rg_2026_score)
print("Possible:", rg_2026_possible)
print("Score %:", rg_2026_score / rg_2026_possible)
print("Pick accuracy:", rg_2026_picks["correct"].mean())
print("Predicted champion:", rg_2026_picks.loc[rg_2026_picks["round"] == "F", "pick_name"].iloc[0])
print("Actual champion:", rg_2026_picks.loc[rg_2026_picks["round"] == "F", "actual_winner_name"].iloc[0])
print("Champion correct:", rg_2026_picks.loc[rg_2026_picks["round"] == "F", "correct"].iloc[0])
print("Optimized expected score:", rg_2026_best_option["expected_score"])

Roland Garros 2026 optimized score: 1260
Possible: 4480
Score %: 0.28125
Pick accuracy: 0.5275590551181102
Predicted champion: Jannik Sinner
Actual champion: Alexander Zverev
Champion correct: False
Optimized expected score: 2286.8741152529183


In [228]:
rg_2026_by_round = rg_2026_picks.groupby("round").agg(
    correct=("correct", "sum"),
    total=("correct", "size"),
    accuracy=("correct", "mean"),
    points=("points_earned", "sum"),
    possible=("points_possible", "sum")
).reset_index()

rg_2026_by_round["score_pct"] = (
    rg_2026_by_round["points"] / rg_2026_by_round["possible"]
)

rg_2026_by_round["round_order"] = (
    rg_2026_by_round["round"].map(round_order)
)

rg_2026_by_round = rg_2026_by_round.sort_values("round_order")

rg_2026_by_round

,round,correct,total,accuracy,points,possible,score_pct,round_order
2,R128,44,64,0.68750,440,640,0.68750,1
5,R64,15,32,0.46875,300,640,0.46875,2
4,R32,5,16,0.31250,200,640,0.31250,3
3,R16,2,8,0.25000,160,640,0.25000,4
1,QF,1,4,0.25000,160,640,0.25000,5
6,SF,0,2,0.00000,0,640,0.00000,6
0,F,0,1,0.00000,0,640,0.00000,7


In [229]:
rg_2026_players = get_tournament_player_features(
    match_df,
    "Roland Garros",
    2026
)

rg_2026_champ_probs_df = pd.DataFrame([
    {
        "player_id": pid,
        "player_name": rg_2026_players[pid]["name"],
        "championship_prob": prob
    }
    for pid, prob in rg_2026_champ_probs.items()
]).sort_values("championship_prob", ascending=False)

rg_2026_champ_probs_df.head(15)
rg_2026_champ_probs_df[
    rg_2026_champ_probs_df["player_name"].str.contains("Zverev", case=False, na=False)
]
rg_2026_champ_probs_df.head(20)

,player_id,player_name,championship_prob
104,S0AG,Jannik Sinner,0.600210
50,D643,Novak Djokovic,0.091974
28,Z355,Alexander Zverev,0.086836
62,RH16,Casper Ruud,0.026573
73,MM58,Daniil Medvedev,0.023045
42,DH58,Alex de Minaur,0.020986
13,FB98,Taylor Fritz,0.013442
76,C0AU,Francisco Cerundolo,0.012213
58,PL56,Tommy Paul,0.010509
90,AG37,Felix Auger-Aliassime,0.009141


Wow, Sinner had a 60% chance to win the tournament according to the model

In [238]:
match_df.tail()

,tourney_id,tourney_name,surface,draw_size,tourney_level,indoor,tourney_date,match_num,winner_id,winner_seed,...,w_h2h_matches_pre,w_h2h_win_pct_pre,l_h2h_wins_pre,l_h2h_losses_pre,l_h2h_matches_pre,l_h2h_win_pct_pre,w_h2h_advantage_pre,l_h2h_advantage_pre,w_h2h_weighted_advantage_pre,l_h2h_weighted_advantage_pre
23319,2026-440,'s-Hertogenbosch,Grass,28.0,250,O,2026-06-13,24.0,DH58,2.0,...,5,0.800000,1,4,5,0.200000,0.300000,-0.300000,0.537528,-0.537528
23320,2026-440,'s-Hertogenbosch,Grass,28.0,250,O,2026-06-13,25.0,MM58,3.0,...,4,0.750000,1,3,4,0.250000,0.250000,-0.250000,0.402359,-0.402359
23321,2026-440,'s-Hertogenbosch,Grass,28.0,250,O,2026-06-13,26.0,MQ75,NaN,...,1,0.000000,1,0,1,1.000000,-0.500000,0.500000,-0.346574,0.346574
23322,2026-321,Stuttgart,Grass,28.0,250,O,2026-06-14,27.0,S0S1,1.0,...,3,0.666667,1,2,3,0.333333,0.166667,-0.166667,0.231049,-0.231049
23323,2026-440,'s-Hertogenbosch,Grass,28.0,250,O,2026-06-14,27.0,MQ75,NaN,...,2,0.000000,2,0,2,1.000000,-0.500000,0.500000,-0.549306,0.549306
